# Detector evaluation under the project conventions

**Purpose.** This notebook evaluates the detection layer (Presidio, GLiNER and Presidio + GLiNER hybrids) under `EVALUATION_CONVENTIONS.md` (branch `evaluation-conventions`): the 13 unified labels, the keep/redact policy, `MATCH_THRESHOLD = 1.0`, F2, and the three development metrics. Detector predictions are cached in `presidio_eval_outputs/cache/`, so a re-run only scores. Open questions for the group are tracked in `notebooks/DETECTOR_OPEN_ISSUES.md`.

**Team decisions this notebook follows (2026-11-04)**

| Topic | Decision |
|---|---|
| Titles | "Mr", "Judge" and similar titles are part of the name, so they are inside the PII span. The title rule (`+T`) extends predicted names over a title directly in front of them. CRAPII spans that leave a title out are tolerated, not corrected |
| Views | Both views are reported. `scoped` drops predicted labels a source never annotates (for example DEMOGRAPHIC on CRAPII); `full` scores every redact-label prediction. `scoped` is used to compare models |
| Sources | Every metric is reported per source; models are compared by macro F2 (unweighted mean over sources) |
| Thresholds | The team notebook's procedure: one confidence threshold per candidate and source, swept from 0.00 to 0.95 on dev, highest span F2, ties to the higher threshold. A global, source-agnostic threshold is an open point for the group |
| Leakage | Character leakage and document leakage both count, broken down by gold label |
| Train | With `INCLUDE_TRAIN = True`, train is scored at the thresholds tuned on dev (the team notebook tunes on dev and applies to held-out data). Thresholds swept on train are reported next to them as a check (Section 11) |
| Test | Not touched here. The test split is scored once, after the final choice under these conventions |
| DOB | Kept separate from other dates: a predicted date with a birth cue in front of it ("born", "date of birth", "DOB") is DOB, others are DATE_OTHER. No current gold has DOB; future data will |
| Verifier | The export carries the unified labels and the document key, so the verifier can read our labels next to the document |
| Ignored spans | TAB MISC and QUANTITY spans are excluded from scoring for now (`EXCLUDE_IGNORED_SPANS = True`), pending group confirmation |

Items still open with the group (LOCATION_OTHER and residences, DEMOGRAPHIC, the global threshold) are implemented as the conventions currently state them.

## Setup

The first cell is the **runtime switch**. `RUNTIME = "auto"` detects Google Colab; set `"local"` or `"colab"` to force one.

- **Local**: the project virtual environment (`requirements.txt`) provides every package and nothing is installed. GLiNER uses the GPU when PyTorch can see one, so the environment needs a CUDA build of PyTorch.
- **Colab**: choose a GPU runtime first (Runtime > Change runtime type). The cell mounts Google Drive and installs the pinned Presidio and GLiNER versions. Place the processed data in `COLAB_DRIVE_DIR/data/processed/`. Outputs and prediction caches are written to Drive as well, so an interrupted run resumes after a disconnect.

The second cell imports the libraries used throughout.

In [ ]:
# ---------------- runtime ----------------
RUNTIME = "auto"                                     # "auto" detects Colab; or force "local" / "colab"
COLAB_DRIVE_DIR = "/content/drive/MyDrive/pii-kachu" # Colab only: holds data/processed/; outputs are written here too

import sys
IN_COLAB = RUNTIME == "colab" or (RUNTIME == "auto" and "google.colab" in sys.modules)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q presidio-analyzer==2.2.364 presidio-anonymizer==2.2.364 gliner==0.2.29
    !python -m spacy download en_core_web_lg -q
print("runtime:", "colab" if IN_COLAB else "local")

In [ ]:
import json, re, hashlib, random, collections, time, warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

## 1. Configuration

- **`QUICK_TEST`**: `True` scores 10% of dev (same `SEED` as the other notebooks); `False` scores all of dev.
- **`INCLUDE_TRAIN`**: `True` (with `QUICK_TEST = False`) also scores all of train in Section 11. Thresholds are still tuned on dev only. Test is never used here.
- **Detectors** are defined in Section 4 and cache their predictions in `presidio_eval_outputs/cache/`; the first run computes them (GLiNER on a GPU is much faster). The unions are rebuilt in memory in Section 6 from their parts' caches.
- **Scoring** settings follow `EVALUATION_CONVENTIONS.md` section 2. Tables go to `presidio_eval_outputs/detector_conv/quick/` or `.../full/` (gitignored).

In [ ]:
# ---------------- run size ----------------
QUICK_TEST = True        # True = 10% of dev (smoke test). False = all of dev.
INCLUDE_TRAIN = False    # True (with QUICK_TEST = False) also scores all of train (Section 11)

# ---------------- data ----------------
# Locates data/processed whether the notebook sits in notebooks/ or in the repository root.
# On Colab (runtime switch in Setup) data and outputs live on Google Drive instead.
DATA_DIR = (Path(COLAB_DRIVE_DIR) / "data" / "processed" if IN_COLAB else
            next((p for p in [Path("../data/processed"), Path("./data/processed")] if p.exists()),
                 Path("../data/processed")))
COMBINED_FILE = None     # e.g. DATA_DIR / "combine.json". When set, it is used instead of the split files.
SOURCES = ["crapii", "tab"]          # UNER is intentionally excluded from the baseline
SPLITS  = ["train", "dev", "test"]
print("Reading data from:", DATA_DIR.resolve())

OUTPUT_DIR = (Path(COLAB_DRIVE_DIR) if IN_COLAB else Path(".")) / "presidio_eval_outputs"   # gitignored: predictions and exports must not be committed
CACHE_DIR  = OUTPUT_DIR / "cache"
EXP_DIR    = OUTPUT_DIR / "detector_conv" / ("quick" if QUICK_TEST else "full")   # experiment tables; one folder per run size
EXP_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True); CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Fraction of each split scored here. Test stays untouched until the final choice.
SAMPLE_FRAC = {"dev": 0.1} if QUICK_TEST else {"dev": 1.0, **({"train": 1.0} if INCLUDE_TRAIN else {})}
SEED = 13

# ---------------- models ----------------
RUN_PRESIDIO   = True
RUN_GLINER     = True
RUN_GLINER_RULES = True   # GLiNER + rule-based post-filter (Section 4.4); needs RUN_GLINER
RUN_FUSION     = False     # late-fusion detectors built from the cached predictions above (Section 4.5)
RUN_PHASE1     = False     # GLiNER prompt variants (Section 4.6); needs RUN_GLINER
RUN_PHASE5     = True     # other GLiNER checkpoints (Section 4.6b)
BAKEOFF_USE    = {"nv_base"}   # only the nvidia checkpoint with base prompts is loaded
# Additional models: add a RUN_<MODEL> switch here and register the model in Section 4 (see Section 8).
ANALYSIS_MODEL = "presidio"

# Presidio settings
SPACY_MODEL = "en_core_web_lg"
LANGUAGE    = "en"
PRESIDIO_SCORE_THRESHOLD = 0.35   # applied at evaluation time; Presidio itself runs at 0 and all scores are cached
ADD_CUSTOM_RECOGNIZERS   = False  # False = vanilla Presidio baseline; see Section 4.2

# GLiNER settings (same checkpoint as the earlier TAB EDA; vanilla, no prompt or threshold tuning)
GLINER_MODEL      = "urchade/gliner_multi_pii-v1"
GLINER_REVISION   = "1fcf13e85f4eef5394e1fcd406cf2ca9ea82351d"   # Hugging Face commit; pins the exact weights
GLINER_THRESHOLD  = 0.5    # applied at evaluation time (EDA value); Section 5.5 sweeps it on dev
GLINER_MIN_SCORE  = 0.1    # inference floor: all candidates above it are cached, so the sweep needs no re-run
NV_MIN_SCORE      = 0.05   # floor for nvidia/gliner-PII (4.6b): at 0.1 its best TAB threshold sat on the floor;
                           # at 0.05 dev TAB F2 rose 0.643 -> 0.671 (nv_base_rules+T), other candidates unchanged
GLINER_WINDOW_CHARS, GLINER_OVERLAP_CHARS = 1200, 200   # long documents are split into overlapping windows
GLINER_BATCH_SIZE = 8      # windows per forward pass; lower it if the GPU runs out of memory
DEVICE            = "auto" # "auto" = CUDA if available, else CPU; or force "cuda" / "cpu"

# ---------------- scoring: EVALUATION_CONVENTIONS.md ----------------
# Entity schema and keep/redact policy (conventions section 1)
UNIFIED_LABELS = ["PERSON_PRIVATE", "PERSON_PUBLIC", "EMAIL", "PHONE", "ID_NUM", "STREET_ADDRESS", "DOB",
                  "DATE_OTHER", "LOCATION_OTHER", "URL_PERSONAL", "OTHER_PII", "DEMOGRAPHIC", "ORG"]
KEEP_LABELS = ["PERSON_PUBLIC", "ORG", "DATE_OTHER", "LOCATION_OTHER"]
REDACT_LABELS = ["PERSON_PRIVATE", "EMAIL", "PHONE", "ID_NUM", "STREET_ADDRESS", "DOB",
                 "URL_PERSONAL", "OTHER_PII", "DEMOGRAPHIC"]
assert sorted(KEEP_LABELS + REDACT_LABELS) == sorted(UNIFIED_LABELS)

# Scoring parameters (conventions section 2)
MATCH_THRESHOLD = 1.0                # span coverage required for a match (separate from model confidence)
BETA = 2.0  # F2 favors recall.

# Not specified in the conventions; pending group confirmation
EXCLUDE_IGNORED_SPANS = True         # TAB ignored_spans (MISC, QUANTITY) are not evaluated characters

# Words never propagated on their own by the R4b name-part rule (used by the rule only, not by scoring)
LEAK_IGNORE_TOKENS = {"mr", "mrs", "ms", "miss", "dr", "prof", "judge", "sir", "the", "of", "and",
                      "a", "an", "in", "at", "on", "de", "la", "le", "von", "van"}

# ---------------- experiments ----------------
BASELINES        = ["presidio", "gliner", "gliner_rules"]
THRESHOLDS       = np.round(np.arange(0.0, 1.0, 0.05), 2)   # team notebook's sweep (4.4)
VIEWS            = ("scoped", "full")
SELECT_VIEW      = "scoped"  # view used to tune thresholds and compare models; "full" is reported alongside
COMPETITIVE_BAND = 0.02      # candidates within this much macro F2 of the best are "competitive"
EXPORT_TOP       = 3         # number of candidates exported for the verifier

## 2. Load data

This step reads the combined CRAPII + TAB corpus in the project's unified format (one document per record: `doc_id`, `source`, `text`, `spans` with `start`, `end`, `label`, `attrs`, and `split`; TAB documents also carry `ignored_spans` for MISC and QUANTITY). Records whose source is not in `SOURCES` are dropped and counted.

Two tables are built:

- `doc_df`: one row per document.
- `gold_df`: one row per gold span (a piece of text an annotator marked).

Each gold label is also mapped to a **coarse type** (PERSON, LOCATION, ADDRESS, DATE, ORG, DEMOGRAPHIC, ID_NUM, EMAIL, PHONE, URL, OTHER). Every detector maps its own labels into the same coarse space in Section 4, which is what allows type-aware matching and model comparison. The mapping follows the unified label table agreed in the meeting:

| Unified label | CRAPII origin | TAB origin | Coarse type |
|---|---|---|---|
| `PERSON_PRIVATE` | NAME_STUDENT | PERSON + DIRECT/QUASI | PERSON |
| `PERSON_PUBLIC` | (not labelled) | PERSON + NO_MASK | PERSON |
| `ORG` | (not labelled) | ORG | ORG |
| `LOCATION_OTHER` | (not labelled) | LOC | LOCATION |
| `STREET_ADDRESS` | STREET_ADDRESS | (under LOC) | ADDRESS |
| `DATE_OTHER` | (not labelled) | DATETIME | DATE |
| `EMAIL` / `PHONE` | EMAIL / PHONE_NUM | (not labelled) | EMAIL / PHONE |
| `ID_NUM` | ID_NUM | CODE | ID_NUM |
| `URL_PERSONAL` | URL_PERSONAL | (not labelled) | URL |
| `OTHER_PII` | USERNAME | (not labelled) | OTHER |
| `DEMOGRAPHIC` | (not labelled) | DEM | DEMOGRAPHIC |

Note on `PERSON_PUBLIC`: TAB's `NO_MASK` means the span stays visible under TAB's guidelines. It does not necessarily mean the person is a public figure.

The printed table shows documents, median length, documents containing PII, and span counts per split and source.

In [ ]:
def _read_records(path):
    """Read a JSON array file or a JSONL file.

    JSONL is split on "\n" only. str.splitlines() must not be used here: it also breaks on Unicode
    separators (U+2028, U+2029, U+0085, form feed, ...), which can legitimately appear unescaped
    inside the document text and would cut a record in half.
    """
    raw = Path(path).read_text(encoding="utf-8-sig")
    if raw.lstrip().startswith("["):
        return json.loads(raw)
    records = []
    for i, line in enumerate(raw.split("\n"), start=1):
        if not line.strip():
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as e:
            raise ValueError(f"{Path(path).name}, line {i}: invalid JSON ({e})") from e
    return records

def load_corpus():
    split_files = {s: DATA_DIR / f"{s}.jsonl" for s in SPLITS}
    records = []
    if COMBINED_FILE is not None:
        for i, d in enumerate(_read_records(COMBINED_FILE)):
            if "split" not in d:
                raise ValueError(f"record {i} in {COMBINED_FILE} has no 'split' field")
            d["_line"] = i
            records.append(d)
        origin = str(COMBINED_FILE)
    elif all(p.exists() for p in split_files.values()):
        for s, p in split_files.items():
            for i, d in enumerate(_read_records(p)):
                d["split"] = s          # trust the file name, not the field
                d["_line"] = i
                records.append(d)
        origin = ", ".join(p.name for p in split_files.values())
    else:
        raise FileNotFoundError(f"No split files in {DATA_DIR.resolve()} and COMBINED_FILE is not set. "
                                "Build them with `python -m src.ingest.build` or set COMBINED_FILE.")
    dropped = collections.Counter(d.get("source") for d in records
                                  if d.get("source") not in SOURCES or d["split"] not in SPLITS)
    kept = [d for d in records if d.get("source") in SOURCES and d["split"] in SPLITS]
    print(f"Loaded {len(records)} records from {origin}; kept {len(kept)}"
          + (f"; dropped by source/split filter: {dict(dropped)}" if dropped else ""))
    return kept

ALL_DOCS = load_corpus()
for d in ALL_DOCS:
    d["key"] = f'{d["split"]}|{d["doc_id"]}'
    d.setdefault("spans", [])
DOCS = {s: [d for d in ALL_DOCS if d["split"] == s] for s in SPLITS}
DOC_BY_KEY = {d["key"]: d for d in ALL_DOCS}


unknown_labels = {s["label"] for d in ALL_DOCS for s in d["spans"]} - set(UNIFIED_LABELS)
if unknown_labels:
    raise ValueError(f"labels outside the unified schema: {sorted(unknown_labels)}")

corpus_overview = pd.DataFrame([{
    "split": d["split"], "source": d["source"],
    "redact_spans": sum(s["label"] in REDACT_LABELS for s in d["spans"]),
    "keep_spans": sum(s["label"] in KEEP_LABELS for s in d["spans"]),
} for d in ALL_DOCS])
display(corpus_overview.groupby(["split", "source"]).agg(
    documents=("redact_spans", "size"),
    pii_documents=("redact_spans", lambda x: int((x > 0).sum())),
    redact_spans=("redact_spans", "sum"),
    keep_spans=("keep_spans", "sum")))
display(pd.crosstab([gs["label"] for d in ALL_DOCS for gs in d["spans"]],
                    [d["source"] for d in ALL_DOCS for gs in d["spans"]]).rename_axis(index="label", columns="source"))

## 3. Helpers

`covered_mask` marks the characters covered by a list of spans.

In [ ]:
ignored_by_key = {d["key"]: d.get("ignored_spans", []) or [] for d in ALL_DOCS}

def covered_mask(n, spans):
    m = np.zeros(n, dtype=bool)
    for s in spans:
        m[max(0, s["start"]):min(n, s["end"])] = True
    return m

## 4. Detection models

Every detector is wrapped in the same small interface so the evaluation in Section 5 never depends on which model produced the predictions.

A `Detector` has:

| Field | Meaning |
|---|---|
| `name` | Short identifier used in tables, plots and cache filenames (for example `"presidio"`) |
| `predict` | Function `text -> list of {"start", "end", "entity_type", "score"}` with character offsets into the original text. It should return all candidates down to a low score; filtering happens at evaluation time |
| `entity_map` | Maps the model's own labels to the coarse types defined in Section 2 |
| `config` | Everything that changes the model's output (model name, label set, settings). It is hashed into the cache filename so a changed configuration never reuses stale predictions |
| `default_threshold` | Score cut-off used in the main results; Section 5.5 tunes it on dev |
| `fallback` | Optional function for labels missing from `entity_map` |

`register(detector)` adds it to `DETECTORS`. Section 4.7 then runs every registered detector over the same sampled documents.

In [ ]:
COARSE_TYPES = ["PERSON", "LOCATION", "ADDRESS", "DATE", "ORG", "DEMOGRAPHIC",
                "ID_NUM", "EMAIL", "PHONE", "URL", "OTHER"]

@dataclass
class Detector:
    name: str
    predict: Callable[[str], list]
    entity_map: dict
    config: Any
    default_threshold: float = 0.5
    fallback: Optional[Callable[[str], str]] = None

    def to_coarse(self, entity_type):
        if entity_type in self.entity_map:
            return self.entity_map[entity_type]
        return self.fallback(entity_type) if self.fallback else "OTHER"

    @property
    def cache_path(self):
        tag = hashlib.md5(json.dumps(self.config, sort_keys=True, default=str).encode()).hexdigest()[:8]
        return CACHE_DIR / f"{self.name}_preds_{tag}.jsonl"

DETECTORS = {}
def register(detector):
    bad = {v for v in detector.entity_map.values() if v not in COARSE_TYPES}
    if bad:
        raise ValueError(f"{detector.name}: entity_map uses unknown coarse types {bad}")
    DETECTORS[detector.name] = detector
    print(f"registered detector '{detector.name}' (default threshold {detector.default_threshold}, "
          f"cache {detector.cache_path.name})")

### 4.1 Presidio configuration

Presidio is configured with spaCy `en_core_web_lg` and an explicit spaCy-to-Presidio entity mapping, so the run is reproducible across Presidio versions.

By default Presidio multiplies spaCy `ORG` scores by 0.4, which places them around 0.34, below most thresholds. Here that down-weighting is switched off (`low_score_entity_names` is empty) and the cut-off is controlled at evaluation time through `PRESIDIO_SCORE_THRESHOLD` instead. The printed list of supported entities should include `ORGANIZATION`.

`ADD_CUSTOM_RECOGNIZERS = False` keeps the run a vanilla Presidio baseline. Setting it to `True` adds two example pattern recognizers (ECHR application numbers and social-media handles) that illustrate cheap rule-based gains; any such change produces a separate cache file and must be reported as a separate configuration, not as the baseline.

In [ ]:
if RUN_PRESIDIO:
    from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
    from presidio_analyzer.nlp_engine import NlpEngineProvider

    nlp_config = {
        "nlp_engine_name": "spacy",
        "models": [{"lang_code": LANGUAGE, "model_name": SPACY_MODEL}],
        "ner_model_configuration": {
            "model_to_presidio_entity_mapping": {
                "PER": "PERSON", "PERSON": "PERSON", "NORP": "NRP",
                "FAC": "LOCATION", "LOC": "LOCATION", "GPE": "LOCATION", "LOCATION": "LOCATION",
                "ORG": "ORGANIZATION", "ORGANIZATION": "ORGANIZATION",
                "DATE": "DATE_TIME", "TIME": "DATE_TIME",
            },
            "low_confidence_score_multiplier": 0.4,
            "low_score_entity_names": [],
            "labels_to_ignore": ["CARDINAL", "EVENT", "LANGUAGE", "LAW", "MONEY",
                                 "ORDINAL", "PERCENT", "PRODUCT", "QUANTITY", "WORK_OF_ART"],
        },
    }
    nlp_engine = NlpEngineProvider(nlp_configuration=nlp_config).create_engine()
    analyzer = AnalyzerEngine(nlp_engine=nlp_engine, supported_languages=[LANGUAGE])

    if ADD_CUSTOM_RECOGNIZERS:
        # ECHR application numbers ("no. 36244/06"); vanilla Presidio has no recognizer for these
        analyzer.registry.add_recognizer(PatternRecognizer(
            supported_entity="ECHR_APP_NO",
            patterns=[Pattern("echr_app_no", r"\b\d{1,6}/\d{2}\b", 0.6)],
            context=["application", "applications", "no", "nos"]))
        # Social-media handles such as "@zzadkth"
        analyzer.registry.add_recognizer(PatternRecognizer(
            supported_entity="SOCIAL_HANDLE", patterns=[Pattern("handle", r"(?<!\w)@[A-Za-z0-9_]{3,30}\b", 0.5)]))

    print("Presidio supported entities:", sorted(analyzer.get_supported_entities()))

### 4.2 Presidio detector

Presidio entity types are mapped to the coarse types. Entities without an explicit mapping fall back to `ID_NUM` when their name suggests an identifier (SSN, passport, credit card, IBAN and similar) and to `OTHER` otherwise. The printed dictionary shows the final mapping for every entity Presidio supports in the installed version.

The cache-tag composition is kept identical to the previous version of this notebook, so existing Presidio caches remain valid.

In [ ]:
if RUN_PRESIDIO:
    PRESIDIO_TO_COARSE = {
        "PERSON": "PERSON", "LOCATION": "LOCATION", "DATE_TIME": "DATE", "NRP": "DEMOGRAPHIC",
        "ORGANIZATION": "ORG", "EMAIL_ADDRESS": "EMAIL", "PHONE_NUMBER": "PHONE", "URL": "URL",
        "ECHR_APP_NO": "ID_NUM", "SOCIAL_HANDLE": "OTHER",   # handles correspond to CRAPII USERNAME -> OTHER_PII
    }
    _ID_HINTS = ("SSN", "NUMBER", "LICENSE", "PASSPORT", "CARD", "IBAN", "CRYPTO", "NHS", "ITIN",
                 "IP_ADDRESS", "CODE", "_ID", "AADHAAR", "PAN", "NRIC", "ABN", "ACN", "TFN", "MEDICARE")
    def presidio_fallback(entity_type):
        return "ID_NUM" if any(h in entity_type for h in _ID_HINTS) else "OTHER"

    def presidio_predict(text):
        res = analyzer.analyze(text=text, language=LANGUAGE, score_threshold=0.0)
        return [{"start": r.start, "end": r.end, "entity_type": r.entity_type, "score": round(float(r.score), 4)}
                for r in res]

    presidio = Detector(
        name="presidio",
        predict=presidio_predict,
        entity_map=PRESIDIO_TO_COARSE,
        config=["presidio", SPACY_MODEL, nlp_config, ADD_CUSTOM_RECOGNIZERS],
        default_threshold=PRESIDIO_SCORE_THRESHOLD,
        fallback=presidio_fallback,
    )
    register(presidio)
    print({e: presidio.to_coarse(e) for e in sorted(analyzer.get_supported_entities())})

### 4.3 GLiNER detector

GLiNER (`GLINER_MODEL`, by default `urchade/gliner_multi_pii-v1`, the checkpoint from the earlier TAB EDA) is a zero-shot NER model: it is given natural-language label prompts and returns spans with a score per prompt. This is a vanilla baseline, so prompts and thresholds are not tuned.

**Label prompts.** These are the EDA prompts with two changes. "monetary amount" is dropped, because TAB QUANTITY spans are ignored in this evaluation. Generic prompts for contact and account identifiers (email, phone number, URL, username) are added so the label set covers common direct identifiers. One prompt set is used for every source; prompts are not tailored to any dataset's annotation scheme. "address" maps to `LOCATION`, as in the EDA: TAB has no separate address type, and CRAPII `STREET_ADDRESS` gold accepts `LOCATION` hits. The mapping is provisional until the group agrees on a uniform label set.

**Long documents.** GLiNER reads about 384 tokens at a time, and TAB judgments are far longer. Each document is cut into windows of `GLINER_WINDOW_CHARS` characters that break on whitespace and overlap by `GLINER_OVERLAP_CHARS`. Offsets are shifted back to the full document. To avoid duplicates and entities truncated at a window edge, each window keeps only the spans that *start* in the part of the overlap closer to its own centre. Any remaining cross-window overlaps are resolved by keeping the higher score.

**Scores and thresholds.** The model runs once at `GLINER_MIN_SCORE` and every candidate is cached. GLiNER resolves overlapping candidates greedily by score, and so does the cross-window step above. As a result, filtering the cached output at a threshold *t* gives the same spans as running the model at *t*, and the dev sweep in Section 5.5 is valid without re-running the model. Below `GLINER_MIN_SCORE` the sweep is flat by construction.

The cell prints the device in use and the first predictions on a TAB document as an offset sanity check.

In [ ]:
if RUN_GLINER:
    import torch
    import gliner as gliner_pkg
    from gliner import GLiNER

    GLINER_DEVICE = ("cuda" if torch.cuda.is_available() else "cpu") if DEVICE == "auto" else DEVICE
    if GLINER_DEVICE == "cpu":
        warnings.warn("GLiNER is running on CPU; a full run will be slow. Use a CUDA build of PyTorch "
                      "locally, or a GPU runtime on Colab.")
    gliner_model = GLiNER.from_pretrained(GLINER_MODEL, revision=GLINER_REVISION).to(GLINER_DEVICE).eval()
    print(f"GLiNER {gliner_pkg.__version__} | {GLINER_MODEL} | device: {GLINER_DEVICE}"
          + (f" ({torch.cuda.get_device_name(0)})" if GLINER_DEVICE == "cuda" else ""))

    # Prompt label -> coarse type. The keys are the prompts sent to the model, one set for every source:
    # the EDA prompts without "monetary amount", plus generic contact/account identifiers.
    GLINER_TO_COARSE = {
        "person": "PERSON", "organization": "ORG", "location": "LOCATION",
        "address": "LOCATION",                       # TAB has no address type; CRAPII STREET_ADDRESS accepts LOCATION
        "date": "DATE", "time": "DATE",
        "case number": "ID_NUM", "identification number": "ID_NUM",
        "nationality": "DEMOGRAPHIC", "occupation": "DEMOGRAPHIC", "age": "DEMOGRAPHIC",
        "email": "EMAIL", "phone number": "PHONE", "url": "URL",
        "username": "OTHER",                         # CRAPII USERNAME -> OTHER_PII
    }
    GLINER_PROMPTS = list(GLINER_TO_COARSE)

    _WS = re.compile(r"\s")
    def gliner_windows(text, size=GLINER_WINDOW_CHARS, overlap=GLINER_OVERLAP_CHARS):
        """(start, end) windows covering the whole text, breaking on whitespace, consecutive ones overlapping."""
        out, start, n = [], 0, len(text)
        while start < n:
            end = min(start + size, n)
            if end < n:
                cut = max(text.rfind(c, start + size // 2, end) for c in " \n\t")
                end = cut if cut > start else end
            out.append((start, end))
            if end >= n:
                break
            m = _WS.search(text, end - overlap, end)
            start = m.end() if m else end
        return out

    def gliner_predict(text):
        wins = gliner_windows(text)
        if not wins:
            return []
        results = gliner_model.inference([text[a:b] for a, b in wins], GLINER_PROMPTS, flat_ner=True,
                                         threshold=GLINER_MIN_SCORE, batch_size=GLINER_BATCH_SIZE)
        cands = []
        for i, ((a, b), ents) in enumerate(zip(wins, results)):
            # each window owns the spans starting between the midpoints of its overlaps with its neighbours
            lo = 0 if i == 0 else (a + wins[i - 1][1]) // 2
            hi = len(text) if i == len(wins) - 1 else (wins[i + 1][0] + b) // 2
            for e in ents:
                if lo <= a + e["start"] < hi:
                    cands.append({"start": a + e["start"], "end": a + e["end"], "entity_type": e["label"],
                                  "score": round(float(e["score"]), 4)})
        kept = []                                    # resolve cross-window overlaps greedily by score, like GLiNER
        for c in sorted(cands, key=lambda c: -c["score"]):
            if all(c["end"] <= k["start"] or c["start"] >= k["end"] for k in kept):
                kept.append(c)
        return sorted(kept, key=lambda c: c["start"])

    _GLINER_RAW = {}                             # text -> raw predictions, reused by gliner_rules (4.4)
    def gliner_predict_memo(text):
        if text not in _GLINER_RAW:
            _GLINER_RAW[text] = gliner_predict(text)
        return _GLINER_RAW[text]

    gliner_det = Detector(
        name="gliner",
        predict=gliner_predict_memo,
        entity_map=GLINER_TO_COARSE,
        config=["gliner", gliner_pkg.__version__, GLINER_MODEL, GLINER_REVISION, GLINER_TO_COARSE, GLINER_MIN_SCORE,
                GLINER_WINDOW_CHARS, GLINER_OVERLAP_CHARS],
        default_threshold=GLINER_THRESHOLD,
    )
    register(gliner_det)

    _d = next((d for d in ALL_DOCS if d["source"] == "tab"), ALL_DOCS[0])
    print(f"sanity check on {_d['key']} (first 3000 chars, score >= {GLINER_THRESHOLD}):")
    for p in [p for p in gliner_predict(_d["text"][:3000]) if p["score"] >= GLINER_THRESHOLD][:10]:
        print(f'  {p["entity_type"]:22s} {p["score"]:.2f} | {_d["text"][p["start"]:p["end"]]}')

### 4.4 GLiNER with a rule-based post-filter

`gliner_rules` is the GLiNER output from 4.3 passed through a few surface-form rules. It is registered as a separate detector, so the vanilla `gliner` numbers remain reportable next to it. The model is not run a second time: the raw predictions from 4.3 are reused.

In the smoke test, most of vanilla GLiNER's CRAPII false positives were spans whose surface form cannot be a direct identifier: pronouns ("I", "we", "me") and generic nouns ("customers", "students") tagged `person`, and the words "email" or "link" tagged `email` or `url`. The rules drop:

| Coarse type | Dropped when |
|---|---|
| `PERSON` | the span is a pronoun, or contains no uppercase letter |
| `EMAIL` | the span contains no `@` |
| `URL` | the span has no scheme (`http://`), no `www.` and no dotted domain |

The rules look only at the predicted text, so they apply identically to every source and use no knowledge of any dataset's labels. One known cost: a name typed entirely in lowercase ("john smith") is dropped. The PERSON-versus-public-figure question (non-protected names, literary characters) is out of scope here and is left to the later verification stage.

In [ ]:
if RUN_GLINER and RUN_GLINER_RULES:
    PRONOUNS = {"i", "me", "my", "mine", "myself", "we", "us", "our", "ours", "ourselves",
                "you", "your", "yours", "yourself", "yourselves", "he", "him", "his", "himself",
                "she", "her", "hers", "herself", "they", "them", "their", "theirs", "themselves",
                "it", "its", "itself", "one", "someone", "somebody", "everyone", "everybody"}
    _URL_SHAPE = re.compile(r"https?://|www\.|\w\.[a-z]{2,}", flags=re.I)

    def keep_span(text, coarse):
        t = text.strip()
        if coarse == "PERSON":
            return t.lower() not in PRONOUNS and any(ch.isupper() for ch in t)
        if coarse == "EMAIL":
            return "@" in t
        if coarse == "URL":
            return bool(_URL_SHAPE.search(t))
        return True

    def gliner_rules_predict(text):
        return [p for p in gliner_predict_memo(text)
                if keep_span(text[p["start"]:p["end"]], GLINER_TO_COARSE.get(p["entity_type"], "OTHER"))]

    register(Detector(
        name="gliner_rules",
        predict=gliner_rules_predict,
        entity_map=GLINER_TO_COARSE,
        config=gliner_det.config + ["rules-v1", sorted(PRONOUNS), _URL_SHAPE.pattern],
        default_threshold=GLINER_THRESHOLD,
    ))

### 4.5 Late fusion of cached predictions

A fused detector reads the predictions of detectors registered above; it never runs a model. Each part is cut at its own threshold and mapped to coarse types, and the spans are then merged.

- `merge_union_extent`: overlapping spans are merged into one span covering both. Type and score come from the higher-scoring span, and `from` records which detectors contributed.

Two unions are registered at the default thresholds: Presidio with vanilla GLiNER, and Presidio with `gliner_rules`. Their recall is the ceiling for any hybrid built from these parts.

The fused detector's `config` holds the cache filenames of its parts, so a changed part never reuses a stale fused cache.

In [ ]:
def merge_union_extent(spans):
    """Merge overlapping spans into one covering both; type and score come from the higher-scoring span."""
    out = []
    for s in sorted(spans, key=lambda s: (s["start"], -s["end"])):
        if out and s["start"] < out[-1]["end"]:
            o = out[-1]
            top = o if o["score"] >= s["score"] else s
            out[-1] = {"start": o["start"], "end": max(o["end"], s["end"]), "entity_type": top["entity_type"],
                       "score": top["score"], "from": sorted(set(o["from"]) | set(s["from"]))}
        else:
            out.append(dict(s))
    return out

_KEY_BY_TEXT = {}
def key_of(text):
    """Document key for a text in the current sample (fused detectors look up their parts' predictions by key)."""
    if text not in _KEY_BY_TEXT:
        _KEY_BY_TEXT.update({d["text"]: d["key"] for d in SAMPLED_DOCS})
    return _KEY_BY_TEXT[text]

def fused_detector(name, parts, merge=merge_union_extent):
    """parts: {registered detector name: threshold, or {coarse type: threshold}}. Predictions come from PREDS, so
    parts must be registered first. A per-type threshold above 1 drops that type from the part."""
    def predict(text):
        key, spans = key_of(text), []
        for m, th in parts.items():
            det = DETECTORS[m]
            for p in PREDS[m][key]:
                c = det.to_coarse(p["entity_type"])
                if p["score"] >= (th.get(c, 0.0) if isinstance(th, dict) else th):
                    spans.append({"start": p["start"], "end": p["end"], "entity_type": c, "score": p["score"],
                                  "from": [m]})
        return merge(spans)
    return Detector(
        name=name,
        predict=predict,
        entity_map={c: c for c in COARSE_TYPES},
        config=["fusion-v1", merge.__name__, {m: [DETECTORS[m].cache_path.name, th] for m, th in parts.items()}],
        default_threshold=0.0,
    )

if RUN_FUSION and RUN_PRESIDIO and RUN_GLINER:
    register(fused_detector("union_presidio_gliner",
                            {"presidio": PRESIDIO_SCORE_THRESHOLD, "gliner": GLINER_THRESHOLD}))
    if RUN_GLINER_RULES:
        register(fused_detector("union_presidio_gliner_rules",
                                {"presidio": PRESIDIO_SCORE_THRESHOLD, "gliner_rules": GLINER_THRESHOLD}))

### 4.6 GLiNER prompt variants (off here: `RUN_PHASE1 = False`)

Each prompt variant is a separate detector with its own cache, built by `make_gliner_detector` with the same checkpoint, windowing and cross-window overlap resolution as `gliner` in 4.3. Two additions:

- **`distractors`**: generic non-PII labels sent to the model alongside the PII prompts. With flat NER, every span gets at most one label, so a span the model scores higher as "pronoun" or "field of study" is no longer emitted as a person or organisation. Distractor hits take part in overlap resolution and are then dropped; they never reach the evaluation.
- **`flat_ner`, `window`, `overlap`**: exposed so they can be varied.

Every variant caches its raw output. The rule-based post-filter from 4.4 is applied afterwards as a derived detector (`<name>_rules`, Section 8), so each variant is scored both with and without rules at no extra cost.

**Prompt sets** (one set for every source; generic wording, aligned with the verifier's label definitions in `src/judge/prompt.py`):

| Set | Idea |
|---|---|
| `terse` | The baseline prompts minus the ones that contributed false positives but no true positives |
| `desc` | Each prompt reworded as a short definition ("full name of a person", "identification or case number") |
| `syn` | Union of terse and descriptive wordings: several prompts per coarse type |
| `desc_distract` | `desc` plus generic distractor labels |
| `base_distract` | the baseline prompts from 4.3 plus generic distractor labels |

Finding: descriptive prompts collapse recall on this checkpoint. Even at a 0.1 threshold they find about half the gold PII the short prompts find, because the model was fine-tuned on short labels. Tuning thresholds for descriptive prompts is parked as a possible later avenue; the short prompts win at every threshold.

`PHASE1_DROP` lists the baseline prompts removed in `terse`. They are the prompts that, in a drop-one ablation, removed false positives without losing true positives.

In [ ]:
PHASE1_DROP = ["occupation", "nationality", "time"]          # from a drop-one prompt ablation
DISTRACTORS = ["pronoun", "group of people", "product or software", "field of study", "event", "title of a work"]

PROMPT_SETS = {
    "terse": {k: v for k, v in GLINER_TO_COARSE.items() if k not in PHASE1_DROP},
    "desc": {
        "full name of a person": "PERSON",
        "name of an organization": "ORG",
        "name of a city, region or country": "LOCATION",
        "street address": "LOCATION",                 # as in 4.3: CRAPII STREET_ADDRESS accepts LOCATION
        "calendar date": "DATE",
        "identification or case number": "ID_NUM",
        "age of a person": "DEMOGRAPHIC",
        "email address": "EMAIL",
        "phone number": "PHONE",
        "web address (URL)": "URL",
        "username or online handle": "OTHER",
    },
}
PROMPT_SETS["syn"] = {**PROMPT_SETS["terse"], **PROMPT_SETS["desc"]}
PROMPT_SETS["base"] = dict(GLINER_TO_COARSE)
PHASE1_VARIANTS = {                                  # detector name -> (prompt set, distractors)
    "g_terse": ("terse", []),
    "g_desc": ("desc", []),
    "g_syn": ("syn", []),
    "g_desc_distract": ("desc", DISTRACTORS),
    "g_base_distract": ("base", DISTRACTORS),
}

def gliner_predict_cfg(text, prompts, drop=(), flat_ner=True, window=GLINER_WINDOW_CHARS, overlap=GLINER_OVERLAP_CHARS,
                       model=None, min_score=None):
    """gliner_predict (4.3) with the prompt list, windowing, flat_ner and the model as arguments; labels in `drop` are
    removed after overlap resolution. model=None uses the 4.3 checkpoint."""
    model = gliner_model if model is None else model
    min_score = GLINER_MIN_SCORE if min_score is None else min_score
    wins = gliner_windows(text, size=window, overlap=overlap)
    if not wins:
        return []
    results = model.inference([text[a:b] for a, b in wins], prompts, flat_ner=flat_ner,
                                     threshold=min_score, batch_size=GLINER_BATCH_SIZE)
    cands = []
    for i, ((a, b), ents) in enumerate(zip(wins, results)):
        lo = 0 if i == 0 else (a + wins[i - 1][1]) // 2
        hi = len(text) if i == len(wins) - 1 else (wins[i + 1][0] + b) // 2
        for e in ents:
            if lo <= a + e["start"] < hi:
                cands.append({"start": a + e["start"], "end": a + e["end"], "entity_type": e["label"],
                              "score": round(float(e["score"]), 4)})
    if flat_ner:
        kept = []
        for c in sorted(cands, key=lambda c: -c["score"]):
            if all(c["end"] <= k["start"] or c["start"] >= k["end"] for k in kept):
                kept.append(c)
    else:
        kept = list({(c["start"], c["end"], c["entity_type"]): c for c in cands}.values())
    return sorted((c for c in kept if c["entity_type"] not in drop), key=lambda c: (c["start"], c["end"]))

def make_gliner_detector(name, label_map, distractors=(), flat_ner=True, window=GLINER_WINDOW_CHARS,
                         overlap=GLINER_OVERLAP_CHARS, model=None, model_id=GLINER_MODEL, revision=GLINER_REVISION,
                         min_score=None):
    """model/model_id/revision default to the 4.3 checkpoint (4.6b passes the other checkpoints)."""
    prompts = list(label_map) + [d for d in distractors if d not in label_map]
    floor = GLINER_MIN_SCORE if min_score is None else min_score   # part of the cache key
    def predict(text):
        return gliner_predict_cfg(text, prompts, drop=set(distractors) - set(label_map), flat_ner=flat_ner,
                                  window=window, overlap=overlap, model=model, min_score=floor)
    return Detector(
        name=name,
        predict=predict,
        entity_map=label_map,
        config=["gliner-variant-v1", gliner_pkg.__version__, model_id, revision, label_map,
                sorted(distractors), flat_ner, floor, window, overlap],
        default_threshold=GLINER_THRESHOLD,
    )

if RUN_PHASE1 and RUN_GLINER:
    for _name, (_set, _distract) in PHASE1_VARIANTS.items():
        register(make_gliner_detector(_name, PROMPT_SETS[_set], distractors=_distract))

### 4.6b Other GLiNER checkpoints (only `nv_base` is loaded here)

Two further PII checkpoints are run in place of `urchade/gliner_multi_pii-v1`, each pinned to a Hugging Face commit. Both load with the installed `gliner` package and use the same windowing, overlap resolution and overlap resolution as 4.3. The inference floor is set per checkpoint: `GLINER_MIN_SCORE` (0.1) for knowledgator and `NV_MIN_SCORE` (0.05) for nvidia, whose best TAB threshold sat at 0.1 when the floor was 0.1. A lower floor only adds candidates (overlap resolution is greedy by score), so every threshold at or above the old floor is unchanged.

| Short name | Checkpoint | Size | License |
|---|---|---|---|
| `know` | `knowledgator/gliner-pii-base-v1.0` | 166M parameters | Apache-2.0 |
| `nv` | `nvidia/gliner-PII` | 445M parameters | NVIDIA Open Model License |

Each checkpoint gets two prompt sets, both covering the same concepts and coarse types as the 4.3 prompts:

- `base`: the 4.3 prompts unchanged.
- `native`: the same concepts in the label style of the checkpoint's model card. knowledgator documents labels such as "name", "email address" and "location city"; nvidia's examples use snake_case ("phone_number", "user_name"). Wording matters for this model family, so the winning wording for one checkpoint may not carry over to another.

In [ ]:
BAKEOFF = {                                     # short name -> (checkpoint, pinned Hugging Face commit)
    "know": ("knowledgator/gliner-pii-base-v1.0", "61726e0ad791dcab3e29339bbec3ad42ded65641"),
    "nv": ("nvidia/gliner-PII", "bd23e8ef4425fd04e34c5204ab49ffaa706eae79"),
}
NATIVE_PROMPTS = {
    "know": {"name": "PERSON", "organization": "ORG", "location city": "LOCATION", "location country": "LOCATION",
             "location address": "LOCATION", "date": "DATE", "time": "DATE", "case number": "ID_NUM",
             "identification number": "ID_NUM", "nationality": "DEMOGRAPHIC", "occupation": "DEMOGRAPHIC",
             "age": "DEMOGRAPHIC", "email address": "EMAIL", "phone number": "PHONE", "url": "URL",
             "username": "OTHER"},
    "nv": {"person": "PERSON", "organization": "ORG", "location": "LOCATION", "street_address": "LOCATION",
           "date": "DATE", "time": "DATE", "case_number": "ID_NUM", "identification_number": "ID_NUM",
           "nationality": "DEMOGRAPHIC", "occupation": "DEMOGRAPHIC", "age": "DEMOGRAPHIC", "email": "EMAIL",
           "phone_number": "PHONE", "url": "URL", "user_name": "OTHER"},
}
BAKEOFF_MODELS, BAKEOFF_VARIANTS = {}, {}       # variant name -> (checkpoint short name, prompt set name)
if RUN_PHASE5 and RUN_GLINER:
    for ck, (repo, rev) in BAKEOFF.items():
        if not any(v.startswith(ck + "_") for v in BAKEOFF_USE):
            continue
        BAKEOFF_MODELS[ck] = GLiNER.from_pretrained(repo, revision=rev).to(GLINER_DEVICE).eval()
        for ps, label_map in (("base", dict(GLINER_TO_COARSE)), ("native", NATIVE_PROMPTS[ck])):
            assert set(label_map.values()) == set(GLINER_TO_COARSE.values()), (ck, ps)
            name = f"{ck}_{ps}"
            if name not in BAKEOFF_USE:
                continue
            register(make_gliner_detector(name, label_map, model=BAKEOFF_MODELS[ck], model_id=repo, revision=rev,
                                          min_score=NV_MIN_SCORE if ck == "nv" else GLINER_MIN_SCORE))
            BAKEOFF_VARIANTS[name] = (ck, ps)

### 4.7 Timing

Each detector's `predict` is wrapped to record seconds and characters for the documents it actually runs on (cached documents are not timed). Totals are merged into `timing.json` by cache filename, so the figure survives later runs that hit the cache. Derived detectors (`gliner_rules`, fused detectors) reuse their parts' predictions; their time includes the model only when the part itself was served from the cache and had to be run inside them.

In [ ]:
TIMING_PATH = EXP_DIR / "timing.json"
TIMING = json.loads(TIMING_PATH.read_text()) if TIMING_PATH.exists() else {}

def timed(det):
    if getattr(det.predict, "_timed", False):
        return
    inner, tag = det.predict, det.cache_path.name
    def predict(text):
        t0 = time.perf_counter()
        out = inner(text)
        t = TIMING.setdefault(tag, {"seconds": 0.0, "chars": 0, "docs": 0})
        t["seconds"] += time.perf_counter() - t0; t["chars"] += len(text); t["docs"] += 1
        return out
    predict._timed = True
    det.predict = predict

for det in DETECTORS.values():
    timed(det)

### 4.8 Run all registered detectors (cached)

Documents are sampled once per split, stratified by source, according to `SAMPLE_FRAC`. Every registered detector runs on exactly the same documents, which keeps the comparison fair.

Predictions are appended to a per-detector cache file in `presidio_eval_outputs/cache/` as they are produced. An interrupted run resumes where it stopped, and raising `SAMPLE_FRAC` later only processes the new documents. Documents that raise an error are recorded in `FAILURES` and skipped rather than stopping the run. Only documents with predictions from **every** registered detector are evaluated.

The printed table shows the number of documents evaluated per split and source.

In [ ]:
def sample_docs():
    rng = random.Random(SEED)
    out = []
    for split in SPLITS:
        frac = SAMPLE_FRAC.get(split, 0)
        if frac <= 0:
            continue
        by_src = collections.defaultdict(list)
        for d in DOCS[split]:
            by_src[d["source"]].append(d)
        for src, lst in sorted(by_src.items()):
            k = len(lst) if frac >= 1 else max(1, round(len(lst) * frac))
            out.extend(rng.sample(lst, k) if k < len(lst) else lst)
    return out

def load_cache(path):
    cache = {}
    if path.exists():
        with path.open(encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    r = json.loads(line); cache[r["key"]] = r["preds"]
    return cache

def run_detector(detector, docs):
    path = detector.cache_path
    cache = load_cache(path)
    todo = [d for d in docs if d["key"] not in cache]
    print(f"[{detector.name}] {len(docs)} docs requested, {len(docs) - len(todo)} cached, {len(todo)} to run -> {path.name}")
    failures = []
    t0 = time.time()
    with path.open("a", encoding="utf-8") as f:
        for d in tqdm(todo, desc=detector.name):
            try:
                preds = detector.predict(d["text"])
            except Exception as e:                       # keep going; record the failure
                failures.append((d["key"], repr(e))); continue
            cache[d["key"]] = preds
            f.write(json.dumps({"key": d["key"], "preds": preds}) + "\n")
    if todo:
        print(f"[{detector.name}] ran {len(todo)} docs in {time.time() - t0:.0f}s; failures: {len(failures)}")
    return {d["key"]: cache[d["key"]] for d in docs if d["key"] in cache}, failures

if not DETECTORS:
    raise RuntimeError("No detectors registered. Set RUN_PRESIDIO (or another RUN_<MODEL> switch) to True.")

SAMPLED_DOCS = sample_docs()
PREDS, FAILURES = {}, {}
for name, det in DETECTORS.items():
    PREDS[name], FAILURES[name] = run_detector(det, SAMPLED_DOCS)

EVAL_DOCS = [d for d in SAMPLED_DOCS if all(d["key"] in PREDS[m] for m in DETECTORS)]
MODELS = list(DETECTORS)
if ANALYSIS_MODEL not in DETECTORS:
    ANALYSIS_MODEL = MODELS[0]
print(f"models: {MODELS}; detailed error analysis uses: {ANALYSIS_MODEL}")
display(pd.Series(collections.Counter((d["split"], d["source"]) for d in EVAL_DOCS), name="docs evaluated").sort_index())

In [ ]:
TIMING_PATH.write_text(json.dumps(TIMING, indent=1))
DEV_DOCS = [d for d in EVAL_DOCS if d["split"] == "dev"]
assert DEV_DOCS, "no dev documents in this run"
print(f"dev documents: {len(DEV_DOCS)}", dict(collections.Counter(d["source"] for d in DEV_DOCS)))

### 4.9 Helpers for derived candidates

`derived_detector` (filtered copy of cached predictions), `transformed_detector` and the surface-form rules (`R4b`, in-document propagation of name parts, is used here), `agree_fused_detector` and `materialise` (fused predictions computed in memory, never cached).

In [ ]:
def derived_detector(name, base, keep, default_threshold=None):
    """Filtered copy of `base`'s predictions; keep(text_of_span, pred, coarse) -> bool."""
    b = DETECTORS[base]
    DETECTORS[name] = Detector(name=name, predict=None, entity_map=b.entity_map, config=b.config + ["derived", name],
                               default_threshold=b.default_threshold if default_threshold is None else default_threshold,
                               fallback=b.fallback)
    PREDS[name] = {k: [p for p in v if keep(DOC_BY_KEY[k]["text"][p["start"]:p["end"]], p, b.to_coarse(p["entity_type"]))]
                   for k, v in PREDS[base].items()}
    return name

def transformed_detector(name, base, fn, rules=()):
    """Predictions = fn(text, coarse-typed predictions of `base`); entity types are coarse types."""
    b = DETECTORS[base]
    DETECTORS[name] = Detector(name=name, predict=None, entity_map={c: c for c in COARSE_TYPES},
                               config=b.config + ["transformed", list(rules)], default_threshold=0.0)
    PREDS[name] = {}
    for k, v in PREDS[base].items():
        typed = [{**p, "entity_type": b.to_coarse(p["entity_type"])} for p in v if p["score"] >= b.default_threshold]
        PREDS[name][k] = fn(DOC_BY_KEY[k]["text"], typed)
    return name

_MONTHS = (r"jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|june?|july?|aug(?:ust)?|sep(?:t(?:ember)?)?"
           r"|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?|monday|tuesday|wednesday|thursday|friday|saturday|sunday")
_DATE_SHAPE = re.compile(rf"\d|\b(?:{_MONTHS})\b", flags=re.I)
_EMAIL_SHAPE = re.compile(r"[^\s@]+@[^\s@]+\.[A-Za-z]{2,}")
FORMAT_CHECKS = {
    "PHONE": lambda t: sum(ch.isdigit() for ch in t) >= 7,
    "ID_NUM": lambda t: any(ch.isdigit() for ch in t),
    "EMAIL": lambda t: bool(_EMAIL_SHAPE.search(t)),
    "URL": lambda t: bool(_URL_SHAPE.search(t)),
}
HONORIFICS = {"mr", "mrs", "ms", "miss", "dr", "prof", "sir", "madam", "judge", "the", "a", "an"}
_LEAD_WORDS = re.compile(r"^(?:(?:" + "|".join(sorted(HONORIFICS, key=len, reverse=True)) + r")\.?\s+)+", flags=re.I)
_LEAD_PUNCT = re.compile(r"^[\s(\[{\"'“‘«-]+")
_TRAIL = re.compile(r"(?:['’]s\b|[\s,;:!?()\[\]{}\"'“”‘’«»-]|\.(?=\s*$)(?<![A-Z]\.))+$")
PROP_MIN_SCORE = 0.9
PROP_TYPES = ("PERSON", "ID_NUM")

def r2_format(text, preds, check_dates=False):
    out = []
    for p in preds:
        t = text[p["start"]:p["end"]]
        c = p["entity_type"]
        if c in FORMAT_CHECKS and not FORMAT_CHECKS[c](t):
            continue
        if check_dates and c == "DATE" and not _DATE_SHAPE.search(t):
            continue
        out.append(p)
    return out

def r3_boundaries(text, preds):
    out = []
    for p in preds:
        st, en = p["start"], p["end"]
        t = text[st:en]
        for rx in (_LEAD_PUNCT, _LEAD_WORDS, _LEAD_PUNCT) if p["entity_type"] in ("PERSON", "ORG") else (_LEAD_PUNCT,):
            m = rx.match(t)
            if m and m.end() < len(t):
                st += m.end(); t = t[m.end():]
        m = _TRAIL.search(t)
        if m and m.start() > 0:
            en -= len(t) - m.start()
        if en > st:
            out.append({**p, "start": st, "end": en})
    return out

def r4_propagate(text, preds, name_parts=False):
    covered = covered_mask(len(text), preds)
    seeds = {}
    for p in preds:
        if p["entity_type"] not in PROP_TYPES or p["score"] < PROP_MIN_SCORE:
            continue
        s = text[p["start"]:p["end"]].strip()
        if len(s) >= 3:
            seeds[s] = max(seeds.get(s, (0, ""))[0], p["score"]), p["entity_type"]
        if name_parts and p["entity_type"] == "PERSON":
            for tok in re.findall(r"[A-Z][\w'’-]{2,}", s):
                if tok.lower() not in HONORIFICS | PRONOUNS | LEAK_IGNORE_TOKENS and tok != s:
                    seeds.setdefault(tok, (p["score"], "PERSON"))
    new = []
    for s, (sc, c) in sorted(seeds.items(), key=lambda kv: -len(kv[0])):
        for m in re.finditer(r"(?<!\w)" + re.escape(s) + r"(?!\w)", text):
            if not covered[m.start():m.end()].any():
                new.append({"start": m.start(), "end": m.end(), "entity_type": c, "score": sc, "from": ["propagated"]})
                covered[m.start():m.end()] = True
    return sorted(preds + new, key=lambda p: (p["start"], p["end"]))

RULES = {
    "R2": lambda t, p: r2_format(t, p),
    "R2b": lambda t, p: r2_format(t, p, check_dates=True),
    "R3": r3_boundaries,
    "R4a": lambda t, p: r4_propagate(t, p),
    "R4b": lambda t, p: r4_propagate(t, p, name_parts=True),
}
RULE_ORDER = ["R2", "R2b", "R3", "R4a", "R4b"]     # filters first, then boundaries, then propagation

def apply_rules(rules):
    order = [r for r in RULE_ORDER if r in rules]
    def fn(text, preds):
        for r in order:
            preds = RULES[r](text, preds)
        return preds
    return fn

def merge_prefer(part):
    """Where spans from different parts overlap, keep `part`'s span and drop the others; non-overlapping spans stay."""
    def merge(spans):
        mine = [s for s in spans if part in s["from"]]
        rest = [s for s in spans if part not in s["from"]
                and all(s["end"] <= m["start"] or s["start"] >= m["end"] for m in mine)]
        return merge_union_extent(mine + rest)      # within a part, overlaps are still merged
    merge.__name__ = f"merge_prefer_{part}"
    return merge

def agree_fused_detector(name, parts, spec, merge=merge_union_extent):
    """fused_detector with agreement-aware thresholds: spec = {part: {'COARSE:agree' | 'COARSE:solo': threshold}}."""
    def predict(text):
        key, spans = key_of(text), []
        for m in parts:
            det = DETECTORS[m]
            others = [(q["start"], q["end"]) for m2 in parts if m2 != m for q in PREDS[m2][key]]
            for p in PREDS[m][key]:
                c = det.to_coarse(p["entity_type"])
                tag = "agree" if any(a < p["end"] and b > p["start"] for a, b in others) else "solo"
                if p["score"] >= spec[m].get(f"{c}:{tag}", 0.0):
                    spans.append({"start": p["start"], "end": p["end"], "entity_type": c, "score": p["score"],
                                  "from": [m]})
        return merge(spans)
    return Detector(name=name, predict=predict, entity_map={c: c for c in COARSE_TYPES},
                    config=["fusion-agree-v1", merge.__name__, {m: [DETECTORS[m].cache_path.name, spec[m]] for m in parts}],
                    default_threshold=0.0)

def materialise(det):
    DETECTORS[det.name] = det
    PREDS[det.name] = {d["key"]: det.predict(d["text"]) for d in EVAL_DOCS}
    return det.name

## 5. Unified labels, titles and dates of birth

**Labels.** Every detector maps its own labels to coarse types (Section 4); this section maps coarse types to the unified labels.

| Coarse type | Unified label | Keep or redact |
|---|---|---|
| PERSON | PERSON_PRIVATE | redact. The detector cannot tell private from public persons, so public persons are left to the verifier |
| ADDRESS | STREET_ADDRESS | redact |
| LOCATION | LOCATION_OTHER | keep |
| DATE | DOB with a birth cue, otherwise DATE_OTHER | DOB redact, DATE_OTHER keep |
| ORG | ORG | keep |
| DEMOGRAPHIC, ID_NUM, EMAIL, PHONE | same name | redact |
| URL | URL_PERSONAL | redact |
| OTHER | OTHER_PII | redact |

**Address fix.** The GLiNER prompts for addresses (`address`, `street_address`, `location address`) used to map to LOCATION, which is now a keep label, so street addresses would never be flagged. They now map to ADDRESS. Only the in-memory label map changes; the cached predictions and their file names stay the same.

**Title rule (`+T`).** A predicted PERSON span is extended to the left over titles directly in front of it (`Mr`, `Mrs`, `Ms`, `Miss`, `Mx`, `Dr`, `Prof`, `Professor`, `Sir`, `Dame`, `Lord`, `Lady`, `Judge`, `Justice`, `Rev`, `Madam`, `Mme`, `Mlle`, `Messrs`, with or without a full stop). Titles must be capitalised, so "I miss John" is not extended. The rule reads only the text, so it applies the same way to every source.

**DOB rule.** A predicted DATE becomes DOB when a birth cue (`born`, `birth`, `birthday`, `DOB`, `D.O.B`) appears within the 40 characters before it, with no full stop or line break in between.

In [ ]:
COARSE_TO_UNIFIED = {"PERSON": "PERSON_PRIVATE", "ADDRESS": "STREET_ADDRESS", "LOCATION": "LOCATION_OTHER",
                     "DATE": "DATE_OTHER", "ORG": "ORG", "DEMOGRAPHIC": "DEMOGRAPHIC", "ID_NUM": "ID_NUM",
                     "EMAIL": "EMAIL", "PHONE": "PHONE", "URL": "URL_PERSONAL", "OTHER": "OTHER_PII"}
assert set(COARSE_TO_UNIFIED) == set(COARSE_TYPES) and set(COARSE_TO_UNIFIED.values()) <= set(UNIFIED_LABELS)

_BIRTH_CUE = re.compile(r"\b(?:born|birth|birthday|DOB|D\.O\.B)\b[^.\n]{0,25}$", flags=re.I)
def unified_label(text, pred, coarse):
    if coarse == "DATE":
        return "DOB" if _BIRTH_CUE.search(text, max(0, pred["start"] - 40), pred["start"]) else "DATE_OTHER"
    return COARSE_TO_UNIFIED[coarse]

# Address prompts -> ADDRESS (in-memory label map only; cache file names do not depend on it)
ADDRESS_PROMPTS = {"address", "street_address", "location address"}
for det in DETECTORS.values():
    hits = ADDRESS_PROMPTS & set(det.entity_map)
    if hits:
        det.entity_map = {**det.entity_map, **{h: "ADDRESS" for h in hits}}
        print(f"{det.name}: {sorted(hits)} -> ADDRESS")

TITLE_WORDS = ["Mr", "Mrs", "Ms", "Miss", "Mx", "Dr", "Prof", "Professor", "Sir", "Dame", "Lord", "Lady",
               "Judge", "Justice", "Rev", "Madam", "Mme", "Mlle", "Messrs"]
_TITLES_BEFORE = re.compile(r"(?:(?<![\w.])(?:" + "|".join(sorted(TITLE_WORDS, key=len, reverse=True))
                            + r")\.?[ \t]+)+$")
def extend_titles(text, preds):
    """PERSON spans grow left over capitalised titles directly in front of them."""
    out = []
    for p in preds:
        if p["entity_type"] == "PERSON":
            m = _TITLES_BEFORE.search(text, max(0, p["start"] - 40), p["start"])
            if m:
                p = {**p, "start": m.start()}
        out.append(p)
    return out

for t, (a, b) in [("The applicant, Mr Henrik Hansen, was born on 3 May 1960.", (18, 31)),
                  ("I miss John a lot.", (7, 11))]:
    print(repr(t[slice(*[(q["start"], q["end"]) for q in extend_titles(t, [{"start": a, "end": b, "entity_type": "PERSON"}])][0])]))
print(unified_label("He was born on 3 May 1960.", {"start": 15}, "DATE"), unified_label("On 3 May 1960 he left.", {"start": 3}, "DATE"))

## 6. Candidates

All candidates are built from cached predictions; no model runs here.

| Candidate | What it is |
|---|---|
| `presidio`, `gliner`, `gliner_rules` | Baselines (vanilla Presidio, urchade GLiNER, urchade GLiNER with the 4.4 post-filter) |
| `nv_base_rules` | `nvidia/gliner-PII` with the base prompts and the 4.4 post-filter |
| `u_presidio_gliner_rules`, `u_presidio_nv_rules` | Presidio ∪ GLiNER at each part's default threshold, overlapping spans merged |
| `nv_union_lowleak+R4b`, `nv_agree_lowleak+R4b` | Presidio ∪ nvidia GLiNER with fixed per-type, per-part thresholds (set earlier to cut leakage under the old 50%-coverage scoring), plus the R4b name-part rule. `union` keeps any span above its part's threshold; `agree` uses one threshold for spans both models found and another for single-model spans. ADDRESS gets the threshold LOCATION had, since addresses used to be LOCATION |

Every candidate also appears with the title rule (`+T`), so its effect can be read directly. The team's threshold sweep (Section 7) is applied on top of each candidate's own scores; for the fused candidates that is the score of the higher-scoring part.

In [ ]:
FROZEN_SPECS = {
 "union_lowleak": {
  "presidio": {
   "PERSON": 0.0,
   "LOCATION": 0.0,
   "ADDRESS": 0.0,
   "DATE": 0.0,
   "ORG": 0.0,
   "DEMOGRAPHIC": 0.9,
   "ID_NUM": 0.0,
   "EMAIL": 0.0,
   "PHONE": 0.0,
   "URL": 0.0,
   "OTHER": 0.0
  },
  "nv_base_rules": {
   "PERSON": 0.3,
   "LOCATION": 0.6,
   "ADDRESS": 0.0,
   "DATE": 0.65,
   "ORG": 0.95,
   "DEMOGRAPHIC": 1.01,
   "ID_NUM": 0.3,
   "EMAIL": 0.25,
   "PHONE": 0.25,
   "URL": 0.35,
   "OTHER": 0.3
  }
 },
 "agree_lowleak": {
  "presidio": {
   "PERSON:agree": 0.0,
   "PERSON:solo": 0.9,
   "LOCATION:agree": 0.0,
   "LOCATION:solo": 0.0,
   "ADDRESS:agree": 0.0,
   "ADDRESS:solo": 0.0,
   "DATE:agree": 0.0,
   "DATE:solo": 0.9,
   "ORG:agree": 0.0,
   "ORG:solo": 0.0,
   "DEMOGRAPHIC:agree": 0.9,
   "DEMOGRAPHIC:solo": 0.9,
   "ID_NUM:agree": 0.0,
   "ID_NUM:solo": 0.35,
   "EMAIL:agree": 0.0,
   "EMAIL:solo": 0.0,
   "PHONE:agree": 0.0,
   "PHONE:solo": 0.0,
   "URL:agree": 0.0,
   "URL:solo": 0.0,
   "OTHER:agree": 0.0,
   "OTHER:solo": 0.0
  },
  "nv_base_rules": {
   "PERSON:agree": 0.0,
   "PERSON:solo": 0.55,
   "LOCATION:agree": 0.6,
   "LOCATION:solo": 0.85,
   "ADDRESS:agree": 0.0,
   "ADDRESS:solo": 0.0,
   "DATE:agree": 0.65,
   "DATE:solo": 0.8,
   "ORG:agree": 1.01,
   "ORG:solo": 0.95,
   "DEMOGRAPHIC:agree": 1.01,
   "DEMOGRAPHIC:solo": 1.01,
   "ID_NUM:agree": 0.0,
   "ID_NUM:solo": 0.3,
   "EMAIL:agree": 0.25,
   "EMAIL:solo": 0.0,
   "PHONE:agree": 0.0,
   "PHONE:solo": 0.25,
   "URL:agree": 0.35,
   "URL:solo": 0.0,
   "OTHER:agree": 0.3,
   "OTHER:solo": 0.35
  }
 }
}   # fixed per-type thresholds for the two nvidia hybrids

derived_detector("nv_base_rules", "nv_base", lambda t, p, c: keep_span(t, c))
CANDIDATE_BASES = [m for m in BASELINES + ["nv_base_rules"] if m in DETECTORS]
CANDIDATE_BASES.append(materialise(fused_detector(
    "u_presidio_gliner_rules", {"presidio": PRESIDIO_SCORE_THRESHOLD, "gliner_rules": GLINER_THRESHOLD})))
CANDIDATE_BASES.append(materialise(fused_detector(
    "u_presidio_nv_rules", {"presidio": PRESIDIO_SCORE_THRESHOLD, "nv_base_rules": GLINER_THRESHOLD})))

for fin, spec in FROZEN_SPECS.items():
    spec = {part: dict(th) for part, th in spec.items()}
    agree = any(":" in k for th in spec.values() for k in th)
    for th in spec.values():
        for tag in ((":agree", ":solo") if agree else ("",)):
            th[f"ADDRESS{tag}"] = th[f"LOCATION{tag}"]
    base = f"nv_{fin}"
    materialise(agree_fused_detector(base, list(spec), spec) if agree else fused_detector(base, spec))
    CANDIDATE_BASES.append(transformed_detector(f"{base}+R4b", base, apply_rules(("R4b",)), ("R4b",)))

def title_variant(base):
    """Copy of `base` with coarse entity types, all scores kept, and the title rule applied."""
    b, name = DETECTORS[base], f"{base}+T"
    DETECTORS[name] = Detector(name=name, predict=None, entity_map={c: c for c in COARSE_TYPES},
                               config=b.config + ["titles-v1", TITLE_WORDS], default_threshold=b.default_threshold)
    PREDS[name] = {k: extend_titles(DOC_BY_KEY[k]["text"], [{**p, "entity_type": b.to_coarse(p["entity_type"])} for p in v])
                   for k, v in PREDS[base].items()}
    return name

CANDIDATES = []
for m in CANDIDATE_BASES:
    CANDIDATES += [m, title_variant(m)]
print(f"{len(CANDIDATES)} candidates:", CANDIDATES)

## 7. Scoring (EVALUATION_CONVENTIONS.md)

The scoring follows `presidio_pii_eval.ipynb` (team notebook, Section 4) line for line:

- **Gold PII** = spans with a redact label, nested spans flattened longest-first. Characters are non-whitespace; with `EXCLUDE_IGNORED_SPANS`, TAB ignored spans that are not PII are not evaluated.
- **Flagged** = predictions at or above the confidence threshold whose unified label is a redact label (duplicates at the same offsets counted once). Keep-label predictions are not flagged.
- **Span recall**: a gold span counts only if every evaluated character is flagged. **Span precision**: a predicted span counts only if every evaluated character is gold PII.
- **Development metrics**: character over-redaction (share of flagged characters that are not PII), character leakage (share of PII characters left uncovered), document leakage (share of PII documents with at least one span not fully covered).

Two additions, both reported next to the team's numbers rather than replacing them:

- **Views.** `full` is the team notebook's scoring. `scoped` also drops flagged predictions whose label the source never annotates (`SCOPE`, from the gold labels of each source), because the gold cannot say whether those are right.
- **Span-level detail** keeps each gold span's evaluated and uncovered characters, so leakage can be broken down by label (Section 9).

`evaluate_document` computes the counts for every document; the per-span and per-prediction rows are built only when `detail=True`, which keeps the threshold sweep fast.

In [ ]:
SCOPE = {src: sorted({s["label"] for d in ALL_DOCS if d["source"] == src for s in d["spans"]} & set(REDACT_LABELS))
         for src in SOURCES}
print("scoped view keeps flagged labels:", SCOPE)

def flatten_longest_first(spans):
    """Count nested or overlapping gold PII spans once: keep the longest, drop spans overlapping it."""
    kept = []
    for span in sorted(spans, key=lambda s: (-(s["end"] - s["start"]), s["start"])):
        if all(span["end"] <= k["start"] or span["start"] >= k["end"] for k in kept):
            kept.append(span)
    return kept

_GOLD_VIEWS = {}
def gold_view(doc):
    """Prediction-independent masks per document, reused across models and thresholds."""
    if doc["key"] not in _GOLD_VIEWS:
        text = doc["text"]
        length = len(text)
        non_whitespace = np.fromiter((not ch.isspace() for ch in text), dtype=bool, count=length)
        redact_spans = [s for s in doc["spans"] if s["label"] in REDACT_LABELS]
        pii = covered_mask(length, redact_spans) & non_whitespace
        evaluated = non_whitespace.copy()
        if EXCLUDE_IGNORED_SPANS:
            evaluated &= ~(covered_mask(length, doc.get("ignored_spans") or []) & ~pii)
        keep = {label: covered_mask(length, [s for s in doc["spans"] if s["label"] == label]) & evaluated & ~pii
                for label in KEEP_LABELS}
        pii_spans = [s for s in flatten_longest_first(redact_spans) if evaluated[s["start"]:s["end"]].any()]
        _GOLD_VIEWS[doc["key"]] = {"evaluated": evaluated, "pii": pii, "keep": keep, "pii_spans": pii_spans}
    return _GOLD_VIEWS[doc["key"]]

_LABELLED = {}
def labelled_predictions(model, doc):
    """The model's predictions with coarse type and unified label, computed once per model and document."""
    k = (model, doc["key"])
    if k not in _LABELLED:
        det, text = DETECTORS[model], doc["text"]
        out = []
        for p in PREDS[model][doc["key"]]:
            c = det.to_coarse(p["entity_type"])
            out.append({**p, "coarse": c, "label": unified_label(text, p, c)})
        _LABELLED[k] = sorted(out, key=lambda p: -p["score"])
    return _LABELLED[k]

def flagged_predictions(preds, threshold, scope=None):
    flagged, seen_offsets = [], set()
    for p in preds:                              # already sorted by descending score
        if (p["score"] < threshold or p["label"] not in REDACT_LABELS or (p["start"], p["end"]) in seen_offsets
                or (scope is not None and p["label"] not in scope)):
            continue
        seen_offsets.add((p["start"], p["end"]))
        flagged.append(p)
    return flagged

def evaluate_document(doc, flagged, detail=False):
    gold = gold_view(doc)
    evaluated, pii = gold["evaluated"], gold["pii"]
    flagged_mask = covered_mask(len(doc["text"]), flagged) & evaluated
    uncovered_pii = pii & ~flagged_mask
    row = {"flagged_character_count": int(flagged_mask.sum()),
           "flagged_pii_character_count": int((flagged_mask & pii).sum()),
           "annotated_pii_character_count": int(pii.sum()),
           "uncovered_pii_character_count": int(uncovered_pii.sum()),
           "pii_document": bool(pii.any()), "leaking_pii_document": bool(uncovered_pii.any())}
    span_rows, pred_rows, fully = [], [], 0
    for s in gold["pii_spans"]:
        span_evaluated = evaluated[s["start"]:s["end"]]
        n_eval = int(span_evaluated.sum())
        n_cov = int((flagged_mask[s["start"]:s["end"]] & span_evaluated).sum())
        fully_covered = n_cov / n_eval >= MATCH_THRESHOLD
        fully += fully_covered
        if detail:
            span_rows.append({"label": s["label"], "text": doc["text"][s["start"]:s["end"]], "coverage": n_cov / n_eval,
                              "fully_covered": fully_covered, "evaluated_characters": n_eval,
                              "uncovered_characters": n_eval - n_cov})
    n_pred = n_pii_only = 0
    for p in flagged:
        span_evaluated = evaluated[p["start"]:p["end"]]
        if not span_evaluated.any():
            continue
        pii_only = (pii[p["start"]:p["end"]] & span_evaluated).sum() / span_evaluated.sum() >= MATCH_THRESHOLD
        n_pred += 1; n_pii_only += pii_only
        if detail:
            pred_rows.append({"entity_type": p["entity_type"], "label": p["label"], "score": p["score"],
                              "text": doc["text"][p["start"]:p["end"]], "pii_only": bool(pii_only)})
    row.update({"evaluated_pii_span_count": len(gold["pii_spans"]), "fully_covered_pii_span_count": fully,
                "evaluated_predicted_span_count": n_pred, "pii_only_predicted_span_count": n_pii_only})
    if detail:
        remaining = flagged_mask & ~pii      # attribute over-redacted characters to keep labels, then unlabeled
        for label in KEEP_LABELS:
            on_label = remaining & gold["keep"][label]
            row[f"over_redacted_{label}"] = int(on_label.sum())
            remaining &= ~on_label
        row["over_redacted_UNLABELED"] = int(remaining.sum())
    return row, span_rows, pred_rows

def evaluate(docs, model, thresholds, view="scoped", detail=False):
    """thresholds: one number, or {source: number}. Returns document, gold-span and predicted-span tables."""
    rows = {"documents": [], "pii_spans": [], "predicted_spans": []}
    for doc in docs:
        th = thresholds[doc["source"]] if isinstance(thresholds, dict) else thresholds
        tag = {"model": model, "view": view, "confidence_threshold": th, "match_threshold": MATCH_THRESHOLD,
               "split": doc["split"], "source": doc["source"], "key": doc["key"]}
        flagged = flagged_predictions(labelled_predictions(model, doc), th,
                                      SCOPE[doc["source"]] if view == "scoped" else None)
        row, span_rows, pred_rows = evaluate_document(doc, flagged, detail)
        rows["documents"].append({**tag, **row})
        rows["pii_spans"].extend({**tag, **r} for r in span_rows)
        rows["predicted_spans"].extend({**tag, **r} for r in pred_rows)
    return {name: pd.DataFrame(r) for name, r in rows.items()}

def span_f_beta(span_precision, span_recall, beta=BETA):
    if span_recall == 0:
        return 0.0
    if np.isnan(span_precision) or np.isnan(span_recall):
        return np.nan
    return (1 + beta**2) * span_precision * span_recall / (beta**2 * span_precision + span_recall)

HEADLINE_METRICS = ["span_precision", "span_recall", "span_f2"]
DEVELOPMENT_METRICS = ["character_over_redaction_percentage", "character_leakage_percentage", "document_leakage_rate"]
COUNT_COLUMNS = ["flagged_character_count", "flagged_pii_character_count", "annotated_pii_character_count",
                 "uncovered_pii_character_count", "pii_document", "leaking_pii_document", "evaluated_pii_span_count",
                 "fully_covered_pii_span_count", "evaluated_predicted_span_count", "pii_only_predicted_span_count"]

def summarize(documents, by=("model", "view", "source")):
    by = list(by)
    table = documents.groupby(by).agg(confidence_threshold=("confidence_threshold", "first"),
                                      documents=("key", "size"), **{c: (c, "sum") for c in COUNT_COLUMNS})
    table = table.rename(columns={"pii_document": "pii_document_count",
                                  "leaking_pii_document": "leaking_pii_document_count"})
    def ratio(numerator, denominator):
        return table[numerator] / table[denominator].replace(0, np.nan)
    table["span_precision"] = ratio("pii_only_predicted_span_count", "evaluated_predicted_span_count")
    table["span_recall"] = ratio("fully_covered_pii_span_count", "evaluated_pii_span_count")
    table["span_f2"] = [span_f_beta(p, r) for p, r in zip(table.span_precision, table.span_recall)]
    table["character_over_redaction_percentage"] = 100 * (
        table.flagged_character_count - table.flagged_pii_character_count) / table.flagged_character_count.replace(0, np.nan)
    table["character_leakage_percentage"] = 100 * ratio("uncovered_pii_character_count", "annotated_pii_character_count")
    table["document_leakage_rate"] = 100 * ratio("leaking_pii_document_count", "pii_document_count")
    return table.reset_index()

SHOW_COLUMNS = ["confidence_threshold", "documents", "evaluated_pii_span_count", "evaluated_predicted_span_count"] \
               + HEADLINE_METRICS + DEVELOPMENT_METRICS
def show_metrics(table):
    formats = {**{m: "{:.3f}" for m in HEADLINE_METRICS + ["macro_f2"]}, **{m: "{:.1f}" for m in DEVELOPMENT_METRICS},
               "confidence_threshold": "{:.2f}"}
    display(table.style.format({k: v for k, v in formats.items() if k in table.columns}, na_rep="-"))

# Check against the team notebook: vanilla Presidio, full view, Presidio's default threshold
_check = summarize(evaluate(DEV_DOCS, "presidio", PRESIDIO_SCORE_THRESHOLD, view="full")["documents"])
show_metrics(_check.set_index(["model", "view", "source"])[SHOW_COLUMNS])

## 8. Threshold sweep and candidate comparison (dev)

Per candidate and source, the confidence threshold with the highest span F2 on dev in the `SELECT_VIEW` view (ties go to the higher threshold), as in the team notebook. Each candidate is then scored in both views at its tuned thresholds. Thresholds tuned and scored on the same dev documents are optimistic; that is the team procedure for now, and the test split stays untouched until the final choice.

**Comparison.** Candidates are ranked by macro F2 (mean over sources of span F2). A candidate is **Pareto-optimal** when no other candidate is at least as good on all four of macro span precision, macro span recall, macro character leakage and macro document leakage, and strictly better on one. **Competitive** candidates are within `COMPETITIVE_BAND` of the best macro F2. When more than one candidate is competitive, the trade-off table shows each one's differences from the top candidate, per source.

In [ ]:
sweep_parts = []
for m in tqdm(CANDIDATES, desc="threshold sweep"):
    for th in THRESHOLDS:
        sweep_parts.append(summarize(evaluate(DEV_DOCS, m, float(th), view=SELECT_VIEW)["documents"]))
sweep = pd.concat(sweep_parts, ignore_index=True)
sweep.to_csv(EXP_DIR / "conv_sweep_dev.csv", index=False)

tuned_thresholds = (sweep.dropna(subset=["span_f2"])
                    .sort_values(["span_f2", "confidence_threshold"], ascending=[False, False])
                    .groupby(["model", "source"]).head(1)
                    .set_index(["model", "source"]).confidence_threshold)
TUNED = {m: {s: float(tuned_thresholds[(m, s)]) for s in SOURCES if (m, s) in tuned_thresholds} for m in CANDIDATES}
(EXP_DIR / "conv_tuned_thresholds.json").write_text(json.dumps(TUNED, indent=1))

DETAIL = {}
for m in tqdm(CANDIDATES, desc="tuned, both views"):
    for view in VIEWS:
        DETAIL[(m, view)] = evaluate(DEV_DOCS, m, TUNED[m], view=view, detail=True)
RESULTS = {name: pd.concat([r[name] for r in DETAIL.values()], ignore_index=True)
           for name in ("documents", "pii_spans", "predicted_spans")}
tuned = summarize(RESULTS["documents"])
tuned["titles"] = tuned.model.str.endswith("+T")
tuned.to_csv(EXP_DIR / "conv_tuned_dev.csv", index=False)

MACRO_METRICS = HEADLINE_METRICS + DEVELOPMENT_METRICS
macro = tuned.groupby(["model", "view"])[MACRO_METRICS].mean().reset_index().rename(columns={"span_f2": "macro_f2"})

def pareto_flags(t):
    up, down = ["span_precision", "span_recall"], ["character_leakage_percentage", "document_leakage_rate"]
    a = np.c_[t[up].to_numpy(), -t[down].to_numpy()]
    return [not any((a[j] >= a[i]).all() and (a[j] > a[i]).any() for j in range(len(a)) if j != i) for i in range(len(a))]

ranked = []
for view, t in macro.groupby("view"):
    t = t.sort_values("macro_f2", ascending=False).reset_index(drop=True)
    t["rank"] = np.arange(1, len(t) + 1)
    t["pareto"] = pareto_flags(t)
    t["competitive"] = t.macro_f2 >= t.macro_f2.max() - COMPETITIVE_BAND
    ranked.append(t)
ranked = pd.concat(ranked, ignore_index=True)
ranked.to_csv(EXP_DIR / "conv_ranking_dev.csv", index=False)
for view in VIEWS:
    print(f"macro over sources, {view} view, thresholds tuned per source on dev ({SELECT_VIEW} view)")
    show_metrics(ranked[ranked.view == view].set_index("model")[["rank", "pareto", "competitive", "macro_f2"]
                                                               + [m for m in MACRO_METRICS if m != "span_f2"]])

### 8.1 Per source

Every candidate at its tuned thresholds, per source and view.

In [ ]:
show_metrics(tuned.set_index(["view", "source", "model"]).sort_index()[SHOW_COLUMNS])

### 8.2 Trade-offs among competitive candidates

Differences from the top candidate (by macro F2, `SELECT_VIEW` view), per source. Positive is better for precision, recall and F2; negative is better for the leakage and over-redaction columns.

In [ ]:
sel = ranked[ranked.view == SELECT_VIEW]
contenders = list(sel[sel.competitive | sel.pareto].model)
top = sel.model.iloc[0]
per_src = tuned[tuned.view == SELECT_VIEW].set_index(["source", "model"])[["confidence_threshold"] + MACRO_METRICS]
rows = []
for src in SOURCES:
    for m in contenders:
        r = per_src.loc[(src, m)]
        rows.append({"source": src, "model": m, "pareto": bool(sel.set_index("model").pareto[m]),
                     "competitive": bool(sel.set_index("model").competitive[m]), **r.to_dict(),
                     **{f"d_{k}": r[k] - per_src.loc[(src, top), k] for k in MACRO_METRICS}})
tradeoffs = pd.DataFrame(rows)
tradeoffs.to_csv(EXP_DIR / "conv_tradeoffs_dev.csv", index=False)
print(f"top candidate: {top}; {len(contenders)} competitive or Pareto-optimal candidates")
fmt = {**{c: "{:.3f}" for c in HEADLINE_METRICS}, **{c: "{:.1f}" for c in DEVELOPMENT_METRICS},
       **{f"d_{c}": "{:+.3f}" for c in HEADLINE_METRICS}, **{f"d_{c}": "{:+.1f}" for c in DEVELOPMENT_METRICS},
       "confidence_threshold": "{:.2f}"}
display(tradeoffs.style.format(fmt))

### 8.3 Trade-off plots

Per source, at each candidate's tuned threshold (`SELECT_VIEW` view): span precision against span recall, and span F2 against document leakage. Pareto-optimal candidates (on the macro metrics) are blue and labelled; the others are grey.

In [ ]:
PARETO_COLOR, OTHER_COLOR = "#2a78d6", "#a3a29c"
pareto_set = set(sel[sel.pareto].model)
fig, axes = plt.subplots(len(SOURCES), 2, figsize=(13, 5 * len(SOURCES)), squeeze=False)
for row, src in zip(axes, SOURCES):
    t = tuned[(tuned.view == SELECT_VIEW) & (tuned.source == src)]
    for ax, (x, y, xl, yl) in zip(row, [("span_recall", "span_precision", "span recall", "span precision"),
                                        ("document_leakage_rate", "span_f2", "document leakage rate (%)", "span F2")]):
        for is_p, color, lab in ((False, OTHER_COLOR, "other candidates"), (True, PARETO_COLOR, "Pareto-optimal (macro)")):
            tt = t[t.model.isin(pareto_set) == is_p]
            ax.scatter(tt[x], tt[y], s=48, color=color, edgecolor="white", linewidth=1.5, label=lab, zorder=3)
            if is_p:
                for r in tt.itertuples():
                    ax.annotate(r.model, (getattr(r, x), getattr(r, y)), fontsize=7, xytext=(4, 3),
                                textcoords="offset points", color="#52514e")
        ax.set(xlabel=xl, ylabel=yl, title=f"{src} | dev | {SELECT_VIEW} view")
        ax.grid(alpha=.25); ax.legend(fontsize=8, frameon=False)
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

### 8.4 Effect of the title rule

Each candidate against its `+T` variant, per source (`SELECT_VIEW` view, each at its own tuned threshold).

In [ ]:
tv = tuned[tuned.view == SELECT_VIEW].assign(base=lambda d: d.model.str.replace(r"\+T$", "", regex=True))
title_effect = (tv.set_index(["base", "source", "titles"])[MACRO_METRICS].unstack("titles"))
title_effect = pd.concat({"without": title_effect.xs(False, axis=1, level=1),
                          "with_titles": title_effect.xs(True, axis=1, level=1)}, axis=1)
for c in MACRO_METRICS:
    title_effect[("change", c)] = title_effect[("with_titles", c)] - title_effect[("without", c)]
display(title_effect["change"].style.format({**{c: "{:+.3f}" for c in HEADLINE_METRICS},
                                             **{c: "{:+.1f}" for c in DEVELOPMENT_METRICS}}))

## 9. Leakage and recall by gold label

For each candidate (tuned thresholds, both views), per source and gold label:

- `span_recall`: share of the label's gold spans fully covered.
- `character_leakage_percentage`: share of the label's PII characters left uncovered.
- `document_leakage_rate`: among documents containing the label, the share where at least one span of that label is not fully covered.
- `share_of_leaking_documents`: among all leaking documents for that candidate and source, the share in which this label leaks. Shares add up to more than 100% when documents leak on several labels.

The full table is in `conv_by_label_dev.csv`; the display shows the competitive and Pareto-optimal candidates.

In [ ]:
ps = RESULTS["pii_spans"]
doc_label = (ps.groupby(["model", "view", "source", "label", "key"])
               .agg(leaks=("fully_covered", lambda x: not x.all())).reset_index())
by_label = ps.groupby(["model", "view", "source", "label"]).agg(
    gold_spans=("fully_covered", "size"), span_recall=("fully_covered", "mean"),
    evaluated_characters=("evaluated_characters", "sum"), uncovered_characters=("uncovered_characters", "sum"))
by_label["character_leakage_percentage"] = 100 * by_label.uncovered_characters / by_label.evaluated_characters
dl = doc_label.groupby(["model", "view", "source", "label"]).agg(documents=("key", "size"), leaking_documents=("leaks", "sum"))
by_label = by_label.join(dl)
by_label["document_leakage_rate"] = 100 * by_label.leaking_documents / by_label.documents
leaking_total = RESULTS["documents"].groupby(["model", "view", "source"]).leaking_pii_document.sum()
by_label["share_of_leaking_documents"] = 100 * by_label.leaking_documents / leaking_total.reindex(
    by_label.index.droplevel("label")).replace(0, np.nan).values
by_label = by_label.reset_index()
by_label.to_csv(EXP_DIR / "conv_by_label_dev.csv", index=False)
show = by_label[by_label.model.isin(contenders) & (by_label.view == SELECT_VIEW)]
display(show.set_index(["source", "label", "model"]).sort_index()
        [["gold_spans", "span_recall", "character_leakage_percentage", "document_leakage_rate", "share_of_leaking_documents"]]
        .style.format({"span_recall": "{:.3f}", "character_leakage_percentage": "{:.1f}",
                       "document_leakage_rate": "{:.1f}", "share_of_leaking_documents": "{:.1f}"}))

### 9.1 What the misses look like

For the top candidate: the most frequent uncovered fragments of partly covered gold spans (characters that stayed visible while others in the span were flagged), per source and label. Useful for spotting boundary problems such as titles or list separators.

In [ ]:
def uncovered_fragments(model, view=SELECT_VIEW, n=10):
    out = collections.defaultdict(collections.Counter)
    for doc in DEV_DOCS:
        gold = gold_view(doc)
        flagged = flagged_predictions(labelled_predictions(model, doc), TUNED[model][doc["source"]],
                                      SCOPE[doc["source"]] if view == "scoped" else None)
        fm = covered_mask(len(doc["text"]), flagged) & gold["evaluated"]
        for s in gold["pii_spans"]:
            ev, cov = gold["evaluated"][s["start"]:s["end"]], fm[s["start"]:s["end"]]
            if cov.any() and (ev & ~cov).any():
                seg = doc["text"][s["start"]:s["end"]]
                left = "".join(ch if (ev[i] and not cov[i]) else " " for i, ch in enumerate(seg)).split()
                out[(doc["source"], s["label"])].update(left)
    return pd.DataFrame([{"source": k[0], "label": k[1], "fragments": ", ".join(f"{w} ({c})" for w, c in v.most_common(n))}
                         for k, v in sorted(out.items())])
display(uncovered_fragments(top))

### 9.2 Over-redaction breakdown and DOB predictions

Share of flagged characters (percent) that fell on each keep label or on unlabeled text, for the contenders (`full` view, as in the team notebook). Then the number of predictions labelled DOB per candidate and source. No current gold has DOB, so every DOB prediction counts against precision in the `full` view and is dropped in the `scoped` view; the counts show how often the rule fires.

In [ ]:
orc = [f"over_redacted_{l}" for l in KEEP_LABELS] + ["over_redacted_UNLABELED"]
od = RESULTS["documents"][RESULTS["documents"].model.isin(contenders) & (RESULTS["documents"].view == "full")]
od = od.groupby(["model", "source"])[["flagged_character_count"] + orc].sum()
over_redaction_breakdown = 100 * od[orc].div(od.flagged_character_count.replace(0, np.nan), axis=0)
over_redaction_breakdown.columns = [c.replace("over_redacted_", "") for c in orc]
display(over_redaction_breakdown.round(1))
over_redaction_breakdown.reset_index().to_csv(EXP_DIR / "conv_over_redaction_dev.csv", index=False)

pr = RESULTS["predicted_spans"]
dob = (pr[(pr.view == "full") & (pr.label == "DOB")].groupby(["model", "source"])
         .agg(dob_predictions=("label", "size"), on_gold_pii=("pii_only", "sum")))
display(dob if len(dob) else "no DOB predictions")

## 10. Export for the verifier

For the `EXPORT_TOP` best contenders by macro F2 that use the title rule (titles are part of the name by team decision): one JSON line per dev document, with every prediction at or above the candidate's tuned threshold for that source. Keep-label predictions are included with `redact: false`, so the verifier sees what the detector chose to leave visible; nothing is dropped by the scoped view. Spans are in text order with ids `s1`, `s2`, ...

| Field | Meaning |
|---|---|
| `key`, `doc_id`, `split`, `source` | The verifier joins on `key` (`<split>|<doc_id>`) to read the document |
| `threshold` | The confidence threshold applied for this document's source |
| `spans[].label` | Unified label (`EVALUATION_CONVENTIONS.md`) |
| `spans[].redact` | `true` for redact labels |
| `spans[].coarse`, `score`, `from` | Coarse type, confidence, and which parts (Presidio, GLiNER, propagation) found the span |

Test documents are not exported until the final run.

In [ ]:
export_models = [m for m in sel.model if m in contenders and m.endswith("+T")][:EXPORT_TOP]   # titles are policy
for m in export_models:
    path = EXP_DIR / f"conv_export_{m.replace('+', '_')}.jsonl"
    n_spans = 0
    with path.open("w", encoding="utf-8") as f:
        for d in DEV_DOCS:
            th = TUNED[m][d["source"]]
            preds = sorted((p for p in labelled_predictions(m, d) if p["score"] >= th), key=lambda p: (p["start"], p["end"]))
            spans = [{"id": f"s{i}", "start": p["start"], "end": p["end"], "text": d["text"][p["start"]:p["end"]],
                      "label": p["label"], "redact": p["label"] in REDACT_LABELS, "coarse": p["coarse"],
                      "score": round(float(p["score"]), 4), "from": p.get("from", [])} for i, p in enumerate(preds, 1)]
            f.write(json.dumps({"key": d["key"], "doc_id": d["doc_id"], "split": d["split"], "source": d["source"],
                                "threshold": th, "spans": spans}, ensure_ascii=False) + "\n")
            n_spans += len(spans)
    print(f"{path.name}: {len(DEV_DOCS)} documents, {n_spans} spans")

## 11. Train split (`INCLUDE_TRAIN = True`)

Every candidate is scored on all of train (CRAPII 18,151 documents, TAB 1,014) in both views, two ways:

- **`dev_tuned`** (reported): the per-source thresholds tuned on dev in Section 8, applied unchanged. This matches the team notebook, which tunes on dev and applies the thresholds to held-out data.
- **`train_swept`** (check): thresholds swept on train with the same rule (highest scoped span F2, ties to the higher threshold). Tuning and scoring on the same documents is optimistic; the gap between the two methods shows how sensitive each result is to the threshold.

Per-document rows are written to `train_docs_<method>.csv.gz`, so any subset of train (another group member's sample, the LLM layer's sample) can be re-summarised with `summarize()` without re-running anything.

**Caveat on the frozen fusion candidates (`nv_union_lowleak…`, `nv_agree_lowleak…`).** Their per-type thresholds were searched on a 10% sample of train under an earlier scoring, so their train numbers are partly in-sample. On the train documents outside that sample they score below `nv_base_rules+T` (CRAPII F2 0.539 vs 0.600), and their TAB document leakage is about 98% because they turn DEMOGRAPHIC off.

In [ ]:
if INCLUDE_TRAIN:
    TRAIN_DOCS = [d for d in EVAL_DOCS if d["split"] == "train"]
    assert TRAIN_DOCS, "INCLUDE_TRAIN needs QUICK_TEST = False"
    print("train documents:", len(TRAIN_DOCS), dict(collections.Counter(d["source"] for d in TRAIN_DOCS)))
    _parts = []
    for m in tqdm(CANDIDATES, desc="train sweep"):
        for th in THRESHOLDS:
            _parts.append(summarize(evaluate(TRAIN_DOCS, m, float(th), view=SELECT_VIEW)["documents"]))
    train_sweep = pd.concat(_parts, ignore_index=True)
    train_sweep.to_csv(EXP_DIR / "train_sweep.csv", index=False)
    _best = (train_sweep.dropna(subset=["span_f2"]).sort_values(["span_f2", "confidence_threshold"], ascending=[False, False])
             .groupby(["model", "source"]).head(1).set_index(["model", "source"]).confidence_threshold)
    TUNED_TRAIN = {m: {s: float(_best[(m, s)]) for s in SOURCES if (m, s) in _best} for m in CANDIDATES}
    (EXP_DIR / "train_tuned_thresholds.json").write_text(json.dumps(TUNED_TRAIN, indent=1))

    TRAIN_DOC_ROWS = {}
    for method, ths in (("dev_tuned", TUNED), ("train_swept", TUNED_TRAIN)):
        TRAIN_DOC_ROWS[method] = pd.concat([evaluate(TRAIN_DOCS, m, ths[m], view=v)["documents"]
                                            for m in tqdm(CANDIDATES, desc=method) for v in VIEWS],
                                           ignore_index=True).assign(method=method)
        TRAIN_DOC_ROWS[method].to_csv(EXP_DIR / f"train_docs_{method}.csv.gz", index=False)
    train_tuned = summarize(pd.concat(TRAIN_DOC_ROWS.values(), ignore_index=True), by=("method", "model", "view", "source"))
    train_tuned.to_csv(EXP_DIR / "train_tuned.csv", index=False)
    train_macro = (train_tuned.groupby(["method", "model", "view"])[MACRO_METRICS].mean().reset_index()
                   .rename(columns={"span_f2": "macro_f2"}))
    train_macro.to_csv(EXP_DIR / "train_macro.csv", index=False)

    _m = train_macro[train_macro.view == SELECT_VIEW].pivot(index="model", columns="method",
                                                            values=["macro_f2", "character_leakage_percentage", "document_leakage_rate"])
    print(f"train, {SELECT_VIEW} view, macro over sources")
    display(_m.sort_values(("macro_f2", "dev_tuned"), ascending=False).style.format("{:.3f}"))
    _p = train_tuned[(train_tuned.view == SELECT_VIEW) & (train_tuned.method == "dev_tuned")]
    show_metrics(_p.set_index(["source", "model"]).sort_index()[SHOW_COLUMNS])
else:
    print("INCLUDE_TRAIN = False: train not scored")

### 11.1 Paired bootstrap between candidates (train, `dev_tuned`)

Documents are resampled with replacement (1,000 times, within each source), and both candidates are scored on the same resample. The table gives the span F2 difference (first minus second) with a 95% percentile interval, per source and for the macro mean. An interval that excludes 0 means the difference is not explained by which documents happened to be in train.

In [ ]:
BOOT_PAIRS = [("nv_base_rules+T", "gliner_rules+T"),               # checkpoint
              ("u_presidio_nv_rules+T", "nv_base_rules+T"),        # adding Presidio to nvidia
              ("u_presidio_gliner_rules+T", "gliner_rules+T")]     # adding Presidio to urchade
BOOT_REPS, BOOT_SEED = 1000, 0
_COUNTS = ["fully_covered_pii_span_count", "evaluated_pii_span_count", "pii_only_predicted_span_count", "evaluated_predicted_span_count"]

def _f2_from_counts(s):
    span_recall, span_precision = s[..., 0] / s[..., 1], s[..., 2] / s[..., 3]
    return np.where(span_recall == 0, 0.0, 5 * span_precision * span_recall / (4 * span_precision + span_recall))

def paired_bootstrap(rows, a, b, reps=BOOT_REPS, seed=BOOT_SEED):
    rng, out, draws = np.random.default_rng(seed), [], {}
    rows = rows[rows.view == SELECT_VIEW]
    for src in SOURCES:
        A = rows[(rows.model == a) & (rows.source == src)].set_index("key")[_COUNTS]
        B = rows[(rows.model == b) & (rows.source == src)].set_index("key").loc[A.index][_COUNTS]
        A, B = A.to_numpy(float), B.to_numpy(float)
        idx = rng.integers(0, len(A), (reps, len(A)))
        draws[src] = _f2_from_counts(A[idx].sum(1)) - _f2_from_counts(B[idx].sum(1))
        out.append({"a": a, "b": b, "source": src, "f2_diff": float(_f2_from_counts(A.sum(0)) - _f2_from_counts(B.sum(0))),
                    "ci_low": np.percentile(draws[src], 2.5), "ci_high": np.percentile(draws[src], 97.5)})
    macro = np.mean([draws[s] for s in SOURCES], axis=0)
    out.append({"a": a, "b": b, "source": "macro", "f2_diff": float(np.mean([r["f2_diff"] for r in out])),
                "ci_low": np.percentile(macro, 2.5), "ci_high": np.percentile(macro, 97.5)})
    return out

if INCLUDE_TRAIN:
    boot = pd.DataFrame([r for a, b in BOOT_PAIRS if a in CANDIDATES and b in CANDIDATES
                         for r in paired_bootstrap(TRAIN_DOC_ROWS["dev_tuned"], a, b)])
    boot.to_csv(EXP_DIR / "train_bootstrap.csv", index=False)
    display(boot.style.format({"f2_diff": "{:+.3f}", "ci_low": "{:+.3f}", "ci_high": "{:+.3f}"}))

### 11.2 CRAPII false positives on train

CRAPII's only name label is `NAME_STUDENT` (mapped to PERSON_PRIVATE in `configs/labels.yaml`). Other private people in an essay, such as teammates, colleagues, interviewees and friends, are not annotated, so a detector that finds them is charged with a false positive. The cell below writes every flagged CRAPII train prediction that is not fully inside gold PII (scoped view, dev-tuned thresholds) with 60 characters of context, to `train_crapii_false_positives.csv.gz`, and counts them by label and by whether they touch gold PII at all.

**Hand-coded sample (70 `nv_base_rules+T` PERSON false positives with no gold overlap, 25% train sample):**

| What the prediction is | Share | Under the conventions |
|---|---|---|
| Private person: teammate, colleague, interviewee, friend, person in user research | ~26% | PERSON_PRIVATE: real PII the gold misses |
| Course instructor or lecturer in the course videos | ~26% | Unclear: public course staff, closer to PERSON_PUBLIC |
| Cited author or public figure | ~34% | PERSON_PUBLIC (keep): a true false positive |
| Not a name ("Team", "Students", "Female", "My friend") | ~14% | True false positive |

Counting the private-person share as correct would raise nvidia's CRAPII span precision from about 0.30 to about 0.45. The sample is small (roughly ±10 points). Some of the "private" names belong to fictional characters in the students' design stories, which a detector cannot tell from real people. Telling a classmate from a cited author depends on context, which is the verifier's job. The output file contains document text: keep it out of git.

In [ ]:
FP_MODELS = ["nv_base_rules+T", "gliner_rules+T", "u_presidio_nv_rules+T", "u_presidio_gliner_rules+T", "presidio"]
if INCLUDE_TRAIN:
    _rows = []
    for m in [m for m in FP_MODELS if m in CANDIDATES]:
        th = TUNED[m]["crapii"]
        for doc in (d for d in TRAIN_DOCS if d["source"] == "crapii"):
            g, text = gold_view(doc), doc["text"]
            for p in flagged_predictions(labelled_predictions(m, doc), th, SCOPE["crapii"]):
                ev = g["evaluated"][p["start"]:p["end"]]
                if not ev.any():
                    continue
                pii_share = (g["pii"][p["start"]:p["end"]] & ev).sum() / ev.sum()
                if pii_share >= MATCH_THRESHOLD:
                    continue                                   # a true positive under the conventions
                _rows.append({"model": m, "key": doc["key"], "label": p["label"], "score": p["score"],
                              "text": text[p["start"]:p["end"]], "overlaps_gold_pii": bool(pii_share > 0),
                              "left": text[max(0, p["start"] - 60):p["start"]], "right": text[p["end"]:p["end"] + 60]})
    crapii_fp = pd.DataFrame(_rows)
    crapii_fp.to_csv(EXP_DIR / "train_crapii_false_positives.csv.gz", index=False)
    display(crapii_fp.groupby(["model", "label", "overlaps_gold_pii"]).size().unstack(fill_value=0))